In [ ]:
# ============================================
# faiss_chroma_experiment.ipynb
#
# [실험 목적]
# 팀 전체 재현성 검증에서 답변 점수가 매번
# 60.30~62.94% 사이로 오르내리는 현상이 관찰됐는데, 이게 검색 엔진
# 자체가 매번 다른 결과를 주기 때문인지, 아니면 검색은 항상 똑같고
# 다른 단계(LLM 생성)에서 변동이 생기는 건지 원인을 가려내는 게 목적
#
# [진행 방식과 알아낸 것]
#
# 1. FAISS 인덱스 별도 구축 (cell 11)
#    - 지금 쓰는 Chroma와 공정하게 비교하기 위해, 똑같은 child 청크와
#      똑같은 임베딩 모델(KURE-v1)로 FAISS 인덱스를 별도로 새로 만듦
#
#
# 2. 같은 질의로 두 엔진 결과 직접 비교 (cell 12~13)
#    - faiss_search() 함수를 정의하고, "학사정보시스템 고도화 사업
#      예산" 같은 질의를 FAISS와 Chroma 양쪽에
#      똑같이 넣어서 top-5 결과(문서, 유사도 점수, 순위)를 나란히
#      출력해서 비교
#
#
# 3. 조건 필터링 포함 하이브리드 검색 비교 (cell 16~)
#    - 메타데이터 조건 필터가 걸린 하이브리드 검색까지 두 엔진에서
#      똑같이 재현해서 비교
#
#
# [알아낸 것 / 결론]
# - FAISS와 Chroma 둘 다 완전히 결정론적이라, 같은 질문을 몇 번을
#   다시 돌려도 항상 똑같은 순위와 점수가 나옴을 확인
# - 즉 팀이 관찰한 "재실행마다 점수가 오르내리는 현상"의 원인은
#   검색 엔진 쪽이 전혀 아니고, gpt-5-mini가 같은 컨텍스트를 받고도
#   매번 조금씩 다른 문장으로 답변을 생성하는 "LLM 생성 단계의
#   변동성" 때문이라는 게 확인됨
# - 이 결론을 근거로 이후 reasoning_effort 조정, 다수결 투표 같은
#   LLM 변동성 완화 실험을 진행하게 됨
# ============================================

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
%cd /content/sprint-public-procurement-rag-assistant
!pwd

/content/sprint-public-procurement-rag-assistant
/content/sprint-public-procurement-rag-assistant


In [3]:
from src.retrieval.indexing import HybridIndex
from src.data_processing.chunking import Chunk

In [4]:
import sys
import types

import src.data_processing.chunking as real_chunking

import pickle
from pathlib import Path

chunking_alias = types.ModuleType('src.chunking')
chunking_alias.Chunk = real_chunking.Chunk
sys.modules['src.chunking'] = chunking_alias

DATA_DIR = Path('/content/drive/MyDrive/중급 프로젝트')

with open(DATA_DIR / 'chunks.pkl', 'rb') as f:
    chunks = pickle.load(f)

print(f"총 청크 수: {len(chunks)}")
print(f"첫 청크 타입: {type(chunks[0])}")

총 청크 수: 18239
첫 청크 타입: <class 'src.data_processing.chunking.Chunk'>


In [5]:
index = HybridIndex(chunks)
print(f"검색 대상(child) chunk 수: {len(index._searchable_chunks)}")
print(f"임베딩 백엔드: {index.embedding_backend.name}")

[HybridIndex] parent 전략 chunk 3664개는 검색 후보에서 제외(context 확장 조회 전용) - 실제 검색 대상 14575개
[embeddings] SentenceTransformer 모델 로드 시도 중... (처음 실행이면 HuggingFace에서 모델을 내려받아 몇 분 걸릴 수 있습니다)


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/220 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/17.2k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

...

[embeddings] SentenceTransformer 사용: nlpai-lab/KURE-v1 (dim=1024)
[HybridIndex] 새로 임베딩 진행: collection=rfp_chunks__nlpai-lab_KURE-v1, backend=nlpai-lab/KURE-v1 (14575개 chunk)


Batches:   0%|          | 0/171 [00:00<?, ?it/s]

Batches:   0%|          | 0/171 [00:00<?, ?it/s]

Batches:   0%|          | 0/115 [00:00<?, ?it/s]

검색 대상(child) chunk 수: 14575
임베딩 백엔드: nlpai-lab/KURE-v1


In [6]:
q = "학사정보시스템 고도화 사업 예산"
hits = index.hybrid_search(q, k=5, expand_to_parent=True)
for h in hits:
    print(f"[{h.matched_by} {h.score:.3f}] {h.doc_id}")
    print(h.text[:200])
    print()

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

[hybrid 0.566] 한영대학_한영대학교 특성화 맞춤형 교육환경 구축 - 트랙운영 학사정보.hwp
[표]
2024년 특성화 맞춤형 교육환경 구축–트랙운영 학사정보시스템 고도화제안요청서

          

2024. 10.

 

[표]
목   차
Ⅰ.사업 안내11. 사업개요12. 추진배경 및 필요성13. 기대효과1Ⅱ.구축 방안21. 구축목표22. 구축일정23. 구축범위3Ⅲ.제안 요청 내용41. 제안 요청 개요4Ⅳ.제안안내 사항51. 입찰 및 계약방법

[hybrid 0.500] 대검찰청_아태 사이버범죄 역량강화 허브(APC-HUB) 홈페이지 및 온라인 교.hwp
○ 온라인 교육 시스템 고도화

[표]
구분 | 구성
PPT교육컨텐츠 내韓‧英스크립트의韓‧英TTS자동 변환시스템 구축(TTS시스템) | -PPT,한쇼,PDF등 교육컨텐츠 내 슬라이드노트(강의상세내용)에입력된 한글 또는 영문 텍스트를 한국어나영어 음성으로 변환되어 나오는TTS를 활용한 자동 변환하는 기능 구축- (관리자 기능 추가) 관리자가 해당 시스템의 설

[hybrid 0.500] 한국어촌어항공단_한국어촌어항공단 경영관리시스템(ERP·GW) 기능 고도.hwp
제 안 요 청 서

[표]
사 업 명 | 한국어촌어항공단차세대 경영관리시스템(ERP·GW)고도화
주관기관 | 한국어촌어항공단

2024.  3.

[표]
담당 | 소속 | 성명 | 전화 | FAX
디지털정보실 | 실장 안경호 | 02-6098-0751 | 02-6098-0739
대리 정재혁 | 02-6098-0754

[표]
목  차
Ⅰ.사업개요11. 일반

[hybrid 0.431] 고려대학교_차세대 포털·학사 정보시스템 구축사업.pdf
제 안 요 청 서 
 
 
고려대학교  
차세대 포털·학사 정보시스템 구축 사업 
 
 
 
 
 
 
 
 
2024. 7. 01 
 
 
 
 
 
 
 
 
※ 본 자료는 제안내용의 설명을 위한 배포자료로, 이외 목적으로 무단복제, 전달 및 사용하는 행위를 금함.

 
 
 
 
 
목   차 

In [7]:
import src.config as config
from pathlib import Path
import src.retrieval.indexing as indexing_module

config.CHROMA_DIR = Path('/content/drive/MyDrive/중급 프로젝트/chroma_db')

indexing_module.CHROMA_DIR = config.CHROMA_DIR

In [8]:
index = HybridIndex(chunks)
print(f"검색 대상(child) chunk 수: {len(index._searchable_chunks)}")
print(f"임베딩 백엔드: {index.embedding_backend.name}")

[HybridIndex] parent 전략 chunk 3664개는 검색 후보에서 제외(context 확장 조회 전용) - 실제 검색 대상 14575개
[embeddings] SentenceTransformer 모델 로드 시도 중... (처음 실행이면 HuggingFace에서 모델을 내려받아 몇 분 걸릴 수 있습니다)


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

[embeddings] SentenceTransformer 사용: nlpai-lab/KURE-v1 (dim=1024)
[HybridIndex] 새로 임베딩 진행: collection=rfp_chunks__nlpai-lab_KURE-v1, backend=nlpai-lab/KURE-v1 (14575개 chunk)


Batches:   0%|          | 0/171 [00:00<?, ?it/s]

Batches:   0%|          | 0/171 [00:00<?, ?it/s]

Batches:   0%|          | 0/115 [00:00<?, ?it/s]

검색 대상(child) chunk 수: 14575
임베딩 백엔드: nlpai-lab/KURE-v1


In [11]:
# FAISS 검색기 준비 (KURE-v1, 기존 방식)
# Chroma와 비교하기 위해 같은 청크(child만) + 같은 임베딩 모델로 별도 구축

import faiss
import numpy as np

child_chunks = index._searchable_chunks

faiss_texts = [c.text for c in child_chunks]
faiss_embeddings = index.embedding_backend.encode(faiss_texts)

index_faiss = faiss.IndexFlatL2(faiss_embeddings.shape[1])
index_faiss.add(np.array(faiss_embeddings).astype('float32'))

faiss_chunk_lookup = child_chunks

print(f"FAISS 인덱스 준비 완료: {index_faiss.ntotal}개 벡터")

Batches:   0%|          | 0/456 [00:00<?, ?it/s]

FAISS 인덱스 준비 완료: 14575개 벡터


In [12]:
def faiss_search(query, k=5):
    query_vec = index.embedding_backend.encode([query])
    distances, indices = index_faiss.search(np.array(query_vec).astype('float32'), k)
    results = []
    for idx, dist in zip(indices[0], distances[0]):
        c = faiss_chunk_lookup[idx]
        results.append({'doc_id': c.doc_id, 'text': c.text, 'score': 1 - dist})
    return results

In [13]:
q = "학사정보시스템 고도화 사업 예산"

print("FAISS")
for r in faiss_search(q, k=5):
    print(f"[{r['score']:.3f}] {r['doc_id']}")

print("\n Chroma")
for h in index.vector_search(q, k=5):
    print(f"[{h.matched_by} {h.score:.3f}] {h.doc_id}")

print("\n Chroma Hybrid")
for h in index.hybrid_search(q, k=5):
    print(f"[{h.matched_by} {h.score:.3f}] {h.doc_id}")

FAISS


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

[0.358] 대검찰청_아태 사이버범죄 역량강화 허브(APC-HUB) 홈페이지 및 온라인 교.hwp
[0.305] 고려대학교_차세대 포털·학사 정보시스템 구축사업.pdf
[0.261] 한국사학진흥재단_대학재정정보시스템(기본재산 및 기채 사후관리) 고.hwp
[0.238] 광주과학기술원_학사시스템 기능개선 사업.hwp
[0.222] 한국연구재단_2024년 대학산학협력활동 실태조사 시스템(UICC) 기능개선.hwp

 Chroma


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

[vector 0.358] 대검찰청_아태 사이버범죄 역량강화 허브(APC-HUB) 홈페이지 및 온라인 교.hwp
[vector 0.305] 고려대학교_차세대 포털·학사 정보시스템 구축사업.pdf
[vector 0.261] 한국사학진흥재단_대학재정정보시스템(기본재산 및 기채 사후관리) 고.hwp
[vector 0.238] 광주과학기술원_학사시스템 기능개선 사업.hwp
[vector 0.222] 한국연구재단_2024년 대학산학협력활동 실태조사 시스템(UICC) 기능개선.hwp

 Chroma Hybrid


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

[hybrid 0.566] 한영대학_한영대학교 특성화 맞춤형 교육환경 구축 - 트랙운영 학사정보.hwp
[hybrid 0.500] 대검찰청_아태 사이버범죄 역량강화 허브(APC-HUB) 홈페이지 및 온라인 교.hwp
[hybrid 0.500] 한국어촌어항공단_한국어촌어항공단 경영관리시스템(ERP·GW) 기능 고도.hwp
[hybrid 0.465] 한영대학_한영대학교 특성화 맞춤형 교육환경 구축 - 트랙운영 학사정보.hwp
[hybrid 0.431] 고려대학교_차세대 포털·학사 정보시스템 구축사업.pdf


In [14]:
# 문서 힌트 추출용 기관명 별칭 / 블랙리스트 / 법률 키워드 맵

import re

ORG_ALIAS_MAP = {
    '대검찰청': ['검찰'],
    '고려대학교': ['고려대'],
    '한국산업단지공단': ['산단'],
    '그랜드코리아레저': ['GKL'],
}

COMMON_SUFFIX_WORDS = {
    '박물관', '시스템', '센터', '공단', '진흥원', '협회', '재단', '연구원', '공사', '대학교',
    '사업', '관리', '운영', '구축', '개선', '개발', '지원', '정보', '용역', '기관', '기술',
    '고도화', '확대', '기능', '서비스', '일자리', '플랫폼', '통합', '접수',
    '일자리재단', '일자리플랫폼', '보험', '입찰공고', '공고',
    '과학연구', '과학연', '학연구', '연구소', '기록관리', '경기기록',
    '학교', '학교 ', ' 학교', '산학협력단', '산학협력', '학협력단',
    '통합시스템',
    '2024년', '2025년',
}
COMMON_FILENAME_WORDS = COMMON_SUFFIX_WORDS | {'용역', '수립', '2차', '1차', '3차', '운영', '및', '구축용역'}

LEGAL_KEYWORDS_MAP = {
    '하도급': ['하도급'],
    '공동수급': ['공동수급', '지분율', '컨소시엄'],
    '지분율': ['지분율', '공동수급'],
    '계약보증금': ['계약보증금', '보증금'],
    '평가': ['배점', '평가비율', '기술평가', '가격평가'],
    '제안서 보상': ['제안서 보상'],
    '불이익': ['부정당업자', '입찰보증금', '귀속'],
    '제출물': ['제출서류', '부', 'USB', '제출규격'],
    '제출': ['제출서류', 'USB'],
    '수량': ['부', 'USB'],
    '구축기간': ['사업기간', '구축기간', '개월'],
    '사업기간': ['사업기간', '구축기간', '개월'],
    '유지보수': ['무상유지보수', '유지보수기간', '하자보수', '무상 하자보수'],
    '참가자격': ['참가자격', '참가 자격'],
    '유지관리': ['하자보수', '유지관리 인력', '무상 하자보수'],
    '교육 의무': ['유지관리 인력', '사용자 및 관리자', '하자보수'],
    '교육을': ['유지관리 인력', '사용자 및 관리자', '하자보수'],
    '검수 후': ['하자보수', '유지관리 인력'],
    '재입찰': ['재입찰', '재공고입찰', '최초의 입찰'],
    '재공고': ['재입찰', '재공고입찰', '최초의 입찰'],
    '조건 변경': ['재입찰', '재공고입찰', '최초의 입찰'],
    '지역 요건': ['주된 영업소', '소재지'],
    '부산에': ['주된 영업소', '소재지'],
    '지역요건': ['주된 영업소', '소재지'],
    '소재지': ['주된 영업소', '소재지'],
    '보유인력': ['보유인력', '배점한도'],
    '배점한도': ['보유인력', '배점한도'],
    '계량평가': ['보유인력', '배점한도', '재무구조'],
    '규모비율': ['규모비율', '환산점수', '점수비중'],
    '환산점수': ['규모비율', '환산점수', '점수비중'],
    '수행실적': ['규모비율', '환산점수', '수행실적'],
    '신인도': ['신인도', '가점'],
    '가점표': ['신인도', '가점'],
    '연구원 승인': ['Lesson', '회람'],
    '발생한 경우': ['Lesson', '회람'],
    '회람': ['Lesson', '회람'],
}

In [15]:
def find_relevant_keywords(question):
    matched = []
    for trigger, kws in LEGAL_KEYWORDS_MAP.items():
        if trigger in question:
            matched.extend(kws)
    return list(set(matched))

def is_aggregation_question(question):
    keywords = ['몇 개', '개수', '다 나열', '몇 건']
    strong_total = '전부' in question or ('총' in question and ('개' in question or '건' in question))
    return any(kw in question for kw in keywords) or strong_total

def extract_filter_conditions(query):
    conditions = {}
    if '억' in query and ('이상' in query or '넘는' in query):
        match = re.search(r'(\d+)억', query)
        if match:
            conditions['금액_최소'] = int(match.group(1)) * 100000000
    if '지자체' in query or '지방자치단체' in query:
        conditions['지자체'] = True
    if '공사' in query and ('OO공사' in query or '발주기관이' in query):
        conditions['공사'] = True
    if 'AI' in query:
        conditions['주제_AI'] = True
    if '긴급' in query:
        conditions['긴급'] = True
    if '보안' in query:
        conditions['보안'] = True
    if '재난' in query:
        conditions['재난'] = True
    return conditions

def is_local_gov(org):
    if org is None or (isinstance(org, float)):
        return False
    return bool(re.search(r'(광역시|특별시|특별자치도|특별자치시|[가-힣]+도|[가-힣]+시|[가-힣]+군|[가-힣]+구)$', str(org).strip()))

def normalize_org_name(name):
    return re.sub(r'(특별시|광역시|특별자치시|특별자치도)', '', name)

In [16]:
# 문서 힌트 추출 (핵심 로직, 최종본)

def extract_doc_hints_multi(question, all_filenames_with_biz):
    q_no_space = question.replace(' ', '').replace('&', '')
    org_candidates = []
    for fname, biz_name in all_filenames_with_biz:
        org_part = fname.replace('refined_', '').split('_')[0].strip()
        org_core = re.sub(r'\s*\(.*?\)\s*', '', org_part).strip()
        org_core_clean = re.sub(r'^\(사\)', '', org_core).strip()
        org_core_clean = re.sub(r'\s*입찰공고\s*$', '', org_core_clean).strip()
        org_core_norm = normalize_org_name(org_core_clean)
        if len(org_core_clean) < 2:
            continue

        matched = False
        if org_core_clean in question:
            matched = True
        elif len(org_core_norm) >= 3 and org_core_norm in question:
            matched = True
        elif org_core_clean in ORG_ALIAS_MAP and any(alias in question for alias in ORG_ALIAS_MAP[org_core_clean]):
            matched = True
        else:
            min_len = 4
            for target_str in [org_core_clean, org_core_norm]:
                for start in range(len(target_str) - min_len + 1):
                    for length in range(len(target_str) - start, min_len - 1, -1):
                        substr = target_str[start:start+length]
                        if substr.strip() in question and substr.strip() not in COMMON_SUFFIX_WORDS:
                            matched = True
                            break
                    if matched:
                        break
                if matched:
                    break
        if matched:
            org_candidates.append((fname, org_core_clean))

    biz_candidates = []
    quoted = re.findall(r"['\"]([^'\"]+)['\"]", question)
    for fname, biz_name in all_filenames_with_biz:
        biz_name = str(biz_name).strip()
        if len(biz_name) >= 4 and biz_name in question:
            biz_candidates.append(fname)
            continue
        for q in quoted:
            if q in biz_name or biz_name in q:
                biz_candidates.append(fname)
                break
        eng_words = re.findall(r'[A-Za-z][A-Za-z&\s]{2,}[A-Za-z]', biz_name)
        for ew in eng_words:
            ew_no_space = ew.strip().replace(' ', '').replace('&', '')
            if len(ew_no_space) >= 4 and ew_no_space in q_no_space:
                biz_candidates.append(fname)
                break

    stopwords_general = {'사업의', '사업에서', '사업은', '어떻게', '되나요', '되나요?', '몇', '어떤', '얼마', '비교', '알려줘', '정리해줘', '무엇인가요', '관련', '입찰공고일', '공고일', '입찰공고'}
    raw_keywords = [w.rstrip('.,?!') for w in re.split(r'[ ,·]', question) if len(w) >= 4]
    keywords_all = [w for w in raw_keywords if w not in stopwords_general and w not in COMMON_FILENAME_WORDS and '입찰공고' not in w]

    def fuzzy_match(kw, text, min_overlap=4):
        kw_ns = kw.replace(' ', '')
        text_ns = text.replace(' ', '')
        if kw_ns in text_ns:
            return True
        for n in range(len(kw_ns), min_overlap - 1, -1):
            if kw_ns[:n] in text_ns:
                return True
        return False

    def keyword_weight(kw):
        return 3 if re.search(r'[A-Za-z]', kw) else 1

    filename_candidates = []
    for fname, biz_name in all_filenames_with_biz:
        fname_clean = fname.replace('refined_', '').replace('.hwp', '').replace('.pdf', '')
        matched_kws = [kw for kw in keywords_all if fuzzy_match(kw, fname_clean)]
        score = sum(keyword_weight(kw) for kw in matched_kws)
        if score > 0:
            filename_candidates.append((fname, score, len(matched_kws)))

    if filename_candidates:
        filename_candidates.sort(key=lambda x: -x[1])
        max_score = filename_candidates[0][1]
        for top_fname, score, cnt in filename_candidates:
            if score >= max_score * 0.6 or score >= 1:
                if top_fname not in [f for f, _ in org_candidates] and top_fname not in biz_candidates:
                    if len(filename_candidates) <= 3 or score >= max(max_score * 0.6, 1):
                        biz_candidates.append(top_fname)

    org_groups = {}
    for fname, org_core in org_candidates:
        org_groups.setdefault(org_core, []).append(fname)

    stopwords = {'사업의', '사업에서', '어떻게', '되나요?', '되나요', '몇', '어떤', '얼마', '비교'}
    keywords = [w for w in re.split(r'[ ,]', question) if len(w) >= 2 and w not in stopwords]

    final_hints = []
    for org_core, fnames in org_groups.items():
        fnames = list(set(fnames))
        if len(fnames) == 1:
            final_hints.append(fnames[0])
        else:
            fname_to_biz = dict(all_filenames_with_biz)
            best_doc, best_score2 = None, -1
            for fname in fnames:
                biz_name = fname_to_biz.get(fname, '')
                score2 = sum(1 for kw in keywords if kw in fname or kw in str(biz_name))
                if score2 > best_score2:
                    best_score2, best_doc = score2, fname
            final_hints.append(best_doc)

    for fname in biz_candidates:
        if fname not in final_hints:
            final_hints.append(fname)

    return list(dict.fromkeys(final_hints))

In [17]:
# 고유 문서 목록 추출 (문서 힌트 매칭용)

seen = set()
all_filenames_with_biz = []
for c in child_chunks:
    if c.doc_id not in seen:
        seen.add(c.doc_id)
        biz = c.metadata.get('발주_기관', '')
        all_filenames_with_biz.append((c.doc_id, biz))

print(f"고유 문서 수: {len(all_filenames_with_biz)}")

고유 문서 수: 98


In [18]:
# 시스템 프롬프트

SYSTEM_PROMPT_NEW_V2 = """
너는 'RFP 챗봇'이야. 입찰메이트 컨설턴트가 제안요청서(RFP) 문서를 빠르게 파악할 수 있게 도와줘.

## 기본 원칙

1. 반드시 아래에 제공된 문서 내용(컨텍스트)에 근거해서만 답변해. 문서에 없는 내용을 추측하거나 지어내지 마.

2. 답변은 간결하고 명확하게 작성해. 불필요한 서론 없이 핵심부터 답해.

3. 질문 유형에 따라 답변 형식을 다르게 해:
   - 단일 사실 조회 (예: "예산이 얼마야?") → 핵심 수치/사실 위주로 간결하게
   - 두 개 이상 비교 (예: "A랑 B 중 뭐가 더 커?") → 각 항목을 나란히 제시하고 비교 결론 제시
   - 목적/배경을 묻는 질문 → 관련 섹션을 요약해서 설명
   - 조건에 맞는 여러 문서를 찾는 질문 → 목록 형태로 정리

4. 이전 대화에서 언급된 문서나 주제가 있으면, 후속 질문("그럼 마감일은?" 등)은 같은 문서/주제 맥락에서 답변해.

5. 답변 끝에는 근거가 된 문서명을 명시해.

## 답변을 거절/기권해야 하는 경우 (매우 중요)

아래 경우에는 문서 안에서 관련 정보를 억지로 찾아서 답하려 하지 말고, 명확히 "답변할 수 없다"고만 말하고 끝내. 관련 있어 보이는 부가 정보를 나열하지 마.

- **범위 밖 요청(out_of_scope)**: 네가 할 수 없는 행동을 요청하는 경우(전화 걸기, 이메일 보내기, 실시간 조회 등), 또는 "오늘", "지금", "최신"처럼 실시간·최신 정보를 요구하는 경우. 이때는 "이 기능은 제가 수행할 수 없습니다" 또는 "실시간 정보는 제공된 문서에서 확인할 수 없습니다"라고만 답하고, 대신 관련 문서를 찾아주거나 연락처를 나열하는 등 다른 시도를 하지 마.

- **근거 부족(insufficient_evidence)**: 낙찰 결과, 경쟁사 현황, 예상 낙찰가처럼 애초에 이 문서(제안요청서)에 있을 수 없는 정보를 물어보는 경우. "확인되지 않습니다"라고만 답해.

- **판단/추측 요청(ambiguous)**: "우리 회사가 자격을 충족하는지 판정해줘", "수주 확률이 얼마냐" 처럼 사용자의 상황과 문서를 대조해서 네가 주관적으로 판단·확률을 계산해야 하는 질문. 이런 판정이나 확률 계산은 네가 할 수 없다고 답하고, 판단에 필요한 조건 목록만 간단히 안내해도 되지만 장황하게 체크리스트를 만들지는 마.

- **사용자가 임의의 가정을 세우고 그 가정으로 확정 답변을 요구하는 경우**: "문서에 없으면 OO라고 가정하고 확정해줘"처럼, 사용자가 제시한 임의의 규칙(추측)을 근거 삼아 사실인 것처럼 답을 만들어달라는 요청. 이건 절대 받아들이지 마. "문서에 없는 정보는 임의로 가정해서 확정할 수 없습니다"라고 답하고, 사용자가 제안한 가정을 그대로 적용해서 계산해주지 마.

## 표 형식 데이터 안내

컨텍스트에 [표]라는 표시와 함께 "항목 | 값" 형태로 된 부분이 나오면, 이는 원본 문서의 표를 옮긴 것이야. 각 줄은 표의 한 행을 의미하고, |로 구분된 각 항목은 표의 열(칸)을 의미해. 이 형식을 참고해서 항목과 값을 정확히 짝지어 답변해.

## 여러 문서 처리

컨텍스트에 여러 문서의 내용이 섞여 있을 수 있어. 각 문서 조각이 어느 문서(파일명)에서 왔는지 구분해서, 서로 다른 문서의 정보를 혼동하거나 섞어서 답하지 마.

일부 정보(예: 긴급 여부, 재공고 여부)는 본문 내용이 아니라 문서명(파일명)에만 표시되어 있을 수 있어. 문서명에 이런 정보가 있으면 그것도 근거로 활용해서 답해.

## 주제/카테고리 판단 시 주의사항

질문의 키워드와 문서 안의 유사한 단어가 겉보기에 비슷해 보여도, 실제 의미는 다를 수 있어. 문서의 실제 사업 목적과 내용까지 확인해서 질문 의도와 정확히 일치하는지 판단하고, 확신이 안 서면 "이 문서는 [실제 의미]를 다루고 있어 질문 의도와 다를 수 있습니다"처럼 구분해서 답해. 단어의 표면적 유사성만으로 포함시키지 마.

## 질문 해석 관련

질문에 "OO", "XX" 같은 placeholder처럼 보이는 표현이 있어도, 이는 실제로 채워야 할 빈칸이 아니라 "특정 패턴을 가진 이름 전체"를 가리키는 일반적인 화법일 수 있어. 예를 들어 "발주기관이 OO공사인 사업"은 "발주기관명이 '공사'로 끝나는 모든 사업"을 뜻하는 것이지, 사용자가 실제 공사명을 지정해줘야 한다는 뜻이 아니야. 이런 경우 되묻지 말고, 컨텍스트 안에서 해당 패턴에 맞는 사업을 최대한 찾아서 답해.

## 금액 표기 관련

금액은 부가세(VAT) 포함/별도 표기가 문서마다 다를 수 있어. 답변할 때 원문에 표기된 형태(포함/별도 여부 포함) 그대로 전달하고, 임의로 환산하지 마.

## 구조화된 필드(공고번호, 사업금액, 입찰 참여 시작일/마감일, 발주기관) 답변 규칙

이 필드들은 컨설턴트의 실제 입찰 결정에 직결되니까 특히 신중하게 답해.

- 검색된 문서 조각과 메타데이터에 명확한 값이 있으면, 근거와 함께 답변해.
- 값이 없거나 불확실하면 절대 추정하지 말고 "확인되지 않습니다"라고 명확히 답해.
- 아래 함정에 특히 주의해:
  - 공고번호를 유사한 다른 번호나 제목의 "[재공고]" 표시만으로 추정하지 마.
  - 개찰 시각이나 제안서 평가 시각을 입찰 참여 마감일로 착각해서 답하지 마. 이 셋은 서로 다른 시점이야.
  - 공개일(공고가 게시된 날짜)을 입찰 참여 시작일로 대체하지 마.
  - 발주기관은 게시기관·수요기관·계약기관이 다를 수 있으니까, 근거 없이 하나를 임의로 선택하지 마.
  - 사업금액이 0원이나 1원으로 보이면, 이건 실제 금액이 아니라 비공개·미확정을 나타내는 표시일 수 있어. 이 경우 실금액처럼 답하지 말고 "금액이 비공개이거나 미확정 상태로 보입니다"라고 답해.

## 참가자격 / 제한조건 / 평가기준 / 제출요건 / 계약 리스크(위약금, 계약보증금 등) 답변 규칙

이 항목들도 컨설턴트가 실제로 입찰 여부를 판단하고 계약 의무를 이해하는 데 직결되니까 신중하게 답해.

- 검색된 문서 조각 안에 명확한 근거가 있을 때만 답변해.
- 명확한 근거가 없으면 "제공된 문서 범위에서는 확인되지 않습니다. 원문 전체 확인이 필요할 수 있습니다"라고 답해.
- 다른 사업의 일반적인 조항이나 통상적인 관행을 이 사업에 적용해서 답하지 마.

## 부분 정보 처리

질문에 여러 정보가 섞여 있고 그중 일부만 확인 가능하면, 확인되는 정보는 근거와 함께 답하고 확인 안 되는 정보만 위 규칙에 따라 "확인되지 않습니다"라고 답해. 일부가 확인 안 된다고 전체 답변을 포기하지 마.

## 컨텍스트 (검색된 문서 조각)
{context}

## 질문
{question}
"""

In [19]:
from google.colab import userdata
import openai

api_key = userdata.get('OPENAI_API_KEY')
client = openai.OpenAI(api_key=api_key)

In [20]:
# 메타데이터 헤더 생성

def meta_header_from_metadata(doc_id, metadata):
    org = metadata.get('발주_기관', '')
    amt = metadata.get('사업_금액')
    amt_str = f"{amt:,.0f}원" if amt not in (None, '') else "확인되지 않음"
    return f"[문서: {doc_id}]\n[발주기관(메타데이터): {org}]\n[사업금액(메타데이터): {amt_str}]"

In [21]:
# 버전 A: FAISS 기반 ask_rfp_final

def ask_rfp_final_faiss(question, model_name="gpt-5-mini", max_retries=2):
    doc_hints = extract_doc_hints_multi(question, all_filenames_with_biz)
    doc_hints = doc_hints[:3]
    keywords = find_relevant_keywords(question)
    conditions = extract_filter_conditions(question)

    # doc_id -> 대표 metadata 딕셔너리 (첫 매칭 청크 기준)
    doc_to_meta = {}
    for c in child_chunks:
        if c.doc_id not in doc_to_meta:
            doc_to_meta[c.doc_id] = c.metadata

    context_parts = []

    def get_doc_chunks(doc_id):
        return [c for c in child_chunks if c.doc_id == doc_id]

    if is_aggregation_question(question) and len(doc_hints) >= 1:
        stopwords_q = {'사업의', '사업에서', '어떻게', '되나요?', '되나요', '몇', '어떤', '얼마', '비교'}
        qkeywords = [w for w in re.split(r'[ ,]', question) if len(w) >= 2 and w not in stopwords_q]
        best_doc, best_score = doc_hints[0], -1
        for fname in doc_hints:
            biz = doc_to_meta.get(fname, {}).get('발주_기관', '')
            score = sum(1 for kw in qkeywords if kw in fname or kw in str(biz))
            if score > best_score:
                best_score, best_doc = score, fname
        doc_hint = best_doc
        header = meta_header_from_metadata(doc_hint, doc_to_meta.get(doc_hint, {}))
        for c in get_doc_chunks(doc_hint):
            context_parts.append(f"{header}\n{c.text}")

    elif len(doc_hints) == 1 and keywords:
        doc_hint = doc_hints[0]
        doc_c = get_doc_chunks(doc_hint)
        keyword_chunks = [c for c in doc_c if any(kw in c.text for kw in keywords)]
        selected = keyword_chunks[:25] if keyword_chunks else faiss_search_filtered(question, k=10, allowed_docs=None)
        header = meta_header_from_metadata(doc_hint, doc_to_meta.get(doc_hint, {}))
        if keyword_chunks:
            for c in selected:
                context_parts.append(f"{header}\n{c.text}")
        else:
            for r in selected:
                context_parts.append(f"{meta_header_from_metadata(r['doc_id'], doc_to_meta.get(r['doc_id'], {}))}\n{r['text']}")

    elif len(doc_hints) >= 2:
        for doc_hint in doc_hints:
            doc_c = get_doc_chunks(doc_hint)
            if keywords:
                matched = [c for c in doc_c if any(kw in c.text for kw in keywords)]
                selected = matched[:8] if matched else doc_c[:8]
            else:
                selected = doc_c[:8]
            header = meta_header_from_metadata(doc_hint, doc_to_meta.get(doc_hint, {}))
            for c in selected:
                context_parts.append(f"{header}\n{c.text}")

    elif doc_hints:
        doc_hint = doc_hints[0]
        doc_c = get_doc_chunks(doc_hint)
        header = meta_header_from_metadata(doc_hint, doc_to_meta.get(doc_hint, {}))
        for c in doc_c[:15]:
            context_parts.append(f"{header}\n{c.text}")

    else:
        results = faiss_search_filtered(question, k=10, allowed_docs=None)
        for r in results:
            context_parts.append(f"{meta_header_from_metadata(r['doc_id'], doc_to_meta.get(r['doc_id'], {}))}\n{r['text']}")

    context = "\n\n---\n\n".join(context_parts)
    final_prompt = SYSTEM_PROMPT_NEW_V2.format(context=context, question=question)

    for attempt in range(max_retries):
        response = client.chat.completions.create(
            model=model_name,
            messages=[{"role": "user", "content": final_prompt}],
            max_completion_tokens=8000,
            reasoning_effort="low"
        )
        answer = response.choices[0].message.content
        if answer:
            return answer
    return "(답변 생성 실패)"

In [22]:
# FAISS 검색 헬퍼 (조건 필터 지원 버전)
def faiss_search_filtered(query, k=10, allowed_docs=None, max_per_doc=3):
    query_vec = index.embedding_backend.encode([query])
    search_k = min(len(child_chunks), 2000)
    distances, indices = index_faiss.search(np.array(query_vec).astype('float32'), search_k)
    seen_docs = {}
    results = []
    for idx in indices[0]:
        c = faiss_chunk_lookup[idx]
        if allowed_docs is not None and c.doc_id not in allowed_docs:
            continue
        count = seen_docs.get(c.doc_id, 0)
        if count < max_per_doc:
            results.append({'doc_id': c.doc_id, 'text': c.text})
            seen_docs[c.doc_id] = count + 1
        if len(results) >= k:
            break
    return results

In [23]:
# 버전 B: Chroma(HybridIndex) 기반 ask_rfp_final

def ask_rfp_final_chroma(question, model_name="gpt-5-mini", max_retries=2):
    doc_hints = extract_doc_hints_multi(question, all_filenames_with_biz)
    doc_hints = doc_hints[:3]
    keywords = find_relevant_keywords(question)
    conditions = extract_filter_conditions(question)

    doc_to_meta = {}
    for c in child_chunks:
        if c.doc_id not in doc_to_meta:
            doc_to_meta[c.doc_id] = c.metadata

    context_parts = []

    def get_doc_chunks(doc_id):
        return [c for c in child_chunks if c.doc_id == doc_id]

    def build_meta_filter(conds):
        if not conds:
            return None
        def _filter(meta):
            if '금액_최소' in conds:
                amt = meta.get('사업_금액')
                if amt is None or amt < conds['금액_최소']:
                    return False
            if conds.get('지자체'):
                if not is_local_gov(meta.get('발주_기관')):
                    return False
            if conds.get('공사'):
                org = str(meta.get('발주_기관', ''))
                if '공사' not in org:
                    return False
            return True
        return _filter

    if is_aggregation_question(question) and len(doc_hints) >= 1:
        stopwords_q = {'사업의', '사업에서', '어떻게', '되나요?', '되나요', '몇', '어떤', '얼마', '비교'}
        qkeywords = [w for w in re.split(r'[ ,]', question) if len(w) >= 2 and w not in stopwords_q]
        best_doc, best_score = doc_hints[0], -1
        for fname in doc_hints:
            biz = doc_to_meta.get(fname, {}).get('발주_기관', '')
            score = sum(1 for kw in qkeywords if kw in fname or kw in str(biz))
            if score > best_score:
                best_score, best_doc = score, fname
        doc_hint = best_doc
        header = meta_header_from_metadata(doc_hint, doc_to_meta.get(doc_hint, {}))
        for c in get_doc_chunks(doc_hint):
            context_parts.append(f"{header}\n{c.text}")

    elif len(doc_hints) == 1 and keywords:
        doc_hint = doc_hints[0]
        doc_c = get_doc_chunks(doc_hint)
        keyword_chunks = [c for c in doc_c if any(kw in c.text for kw in keywords)]
        header = meta_header_from_metadata(doc_hint, doc_to_meta.get(doc_hint, {}))
        if keyword_chunks:
            for c in keyword_chunks[:25]:
                context_parts.append(f"{header}\n{c.text}")
        else:
            hits = index.hybrid_search(question, k=10, expand_to_parent=True)
            for h in hits:
                context_parts.append(f"{meta_header_from_metadata(h.doc_id, doc_to_meta.get(h.doc_id, {}))}\n{h.text}")

    elif len(doc_hints) >= 2:
        for doc_hint in doc_hints:
            doc_c = get_doc_chunks(doc_hint)
            if keywords:
                matched = [c for c in doc_c if any(kw in c.text for kw in keywords)]
                selected = matched[:8] if matched else doc_c[:8]
            else:
                selected = doc_c[:8]
            header = meta_header_from_metadata(doc_hint, doc_to_meta.get(doc_hint, {}))
            for c in selected:
                context_parts.append(f"{header}\n{c.text}")

    elif doc_hints:
        doc_hint = doc_hints[0]
        doc_c = get_doc_chunks(doc_hint)
        header = meta_header_from_metadata(doc_hint, doc_to_meta.get(doc_hint, {}))
        for c in doc_c[:15]:
            context_parts.append(f"{header}\n{c.text}")

    elif conditions:
        meta_filter = build_meta_filter(conditions)
        hits = index.hybrid_search(question, k=80, meta_filter=meta_filter, expand_to_parent=True)
        for h in hits:
            context_parts.append(f"{meta_header_from_metadata(h.doc_id, doc_to_meta.get(h.doc_id, {}))}\n{h.text}")

    else:
        hits = index.hybrid_search(question, k=10, expand_to_parent=True)
        for h in hits:
            context_parts.append(f"{meta_header_from_metadata(h.doc_id, doc_to_meta.get(h.doc_id, {}))}\n{h.text}")

    context = "\n\n---\n\n".join(context_parts)
    final_prompt = SYSTEM_PROMPT_NEW_V2.format(context=context, question=question)

    for attempt in range(max_retries):
        response = client.chat.completions.create(
            model=model_name,
            messages=[{"role": "user", "content": final_prompt}],
            max_completion_tokens=8000,
            reasoning_effort="low"
        )
        answer = response.choices[0].message.content
        if answer:
            return answer
    return "(답변 생성 실패)"

In [36]:
import json

with open('data/golden_set_v3/rag-56.draft.jsonl', encoding='utf-8') as f:
    rag56 = [json.loads(l) for l in f if l.strip()]

print(f"총 문항 수: {len(rag56)}")

총 문항 수: 56


In [37]:
# 56문항 답변 생성 - FAISS 버전

faiss_answers = []
for item in rag56:
    cid = item['case_id']
    task_type = item['task_type']
    question = item['question']

    answer = ask_rfp_final_faiss(question)
    faiss_answers.append({'case_id': cid, 'task_type': task_type, 'question': question, 'answer': answer})
    print(f"[FAISS][{cid}][{task_type}] {question}")
    print(answer)
    print()

[FAISS][supplemental-qa-c01][single_doc] 평택시가 정류장 이용객에게 실시간 운행정보를 제공하려고 추진한 2024년 사업에는 예산이 얼마나 배정됐나요?
￦999,494,600원(부가세 포함)입니다.

근거: 경기도 평택시_2024년도 평택시 버스정보시스템(BIS) 구축사업.hwp

[FAISS][supplemental-qa-c02][single_doc] 평택시의 실시간 버스 도착정보 제공 체계를 새로 만드는 2024년 사업은 착수 후 언제까지 완료해야 하나요?
착수일로부터 2024년 10월 31일까지 완료해야 합니다.

근거: 2024년도 평택시 버스정보시스템(BIS) 구축사업.hwp

[FAISS][supplemental-qa-c03][single_doc] GKL이 임직원 협업과 전자결재 환경을 새로 구축하려는 2024년 정보화 사업에 책정한 금액은 얼마인가요?
1,515,000,000원(사업예산: 1,515,000천원, 부가세 포함). 근거: 그랜드코리아레저(주)_2024년도 GKL  그룹웨어 시스템 구축 용역.hwp

[FAISS][supplemental-qa-c04][single_doc] GKL이 사내 협업 업무 환경을 새로 만드는 사업은 어떤 경쟁 방식과 낙찰 절차로 계약하나요?
- 입찰방식(경쟁 방식): 제한경쟁입찰  
- 낙찰(선정) 절차: 협상에 의한 계약(사업자 선정은 협상에 의함)

근거: 그랜드코리아레저(주)_2024년도 GKL  그룹웨어 시스템 구축 용역.hwp

[FAISS][supplemental-qa-c05][single_doc] 한국농어촌공사가 네팔에서 물 관리 정보를 디지털화하기 위해 추진하는 시범 시스템은 착수 후 얼마 동안 수행하나요?
용역착수일로부터 6개월 동안 수행합니다. 근거: 한국농어촌공사_네팔 수자원관리 정보화사업-Pilot 시스템 구축용역.hwp

[FAISS][supplemental-qa-c06][single_doc] 한국농어촌공사가 네팔의 수자원 정보를 관리할 시범 시스템을 만드는 

In [38]:
# 56문항 답변 생성 - Chroma(HybridIndex) 버전

chroma_answers = []
for item in rag56:
    cid = item['case_id']
    task_type = item['task_type']
    question = item['question']

    answer = ask_rfp_final_chroma(question)
    chroma_answers.append({'case_id': cid, 'task_type': task_type, 'question': question, 'answer': answer})
    print(f"[Chroma][{cid}][{task_type}] {question}")
    print(answer)
    print()

[Chroma][supplemental-qa-c01][single_doc] 평택시가 정류장 이용객에게 실시간 운행정보를 제공하려고 추진한 2024년 사업에는 예산이 얼마나 배정됐나요?
￦999,494,600원(부가세 포함). 근거: 2024년도 평택시 버스정보시스템(BIS) 구축사업 용역 제안요청서 (경기도 평택시_2024년도 평택시 버스정보시스템(BIS) 구축사업.hwp).

[Chroma][supplemental-qa-c02][single_doc] 평택시의 실시간 버스 도착정보 제공 체계를 새로 만드는 2024년 사업은 착수 후 언제까지 완료해야 하나요?
착수일로부터 2024년 10월 31일까지 완료해야 합니다. 근거: 2024년도 평택시 버스정보시스템(BIS) 구축사업 제안요청서 (경기도 평택시).

[Chroma][supplemental-qa-c03][single_doc] GKL이 임직원 협업과 전자결재 환경을 새로 구축하려는 2024년 정보화 사업에 책정한 금액은 얼마인가요?
사업예산: 1,515,000천원 (부가세 포함) — 즉 1,515,000,000원. 근거: 그랜드코리아레저(주)_2024년도 GKL 그룹웨어 시스템 구축 용역.hwp

[Chroma][supplemental-qa-c04][single_doc] GKL이 사내 협업 업무 환경을 새로 만드는 사업은 어떤 경쟁 방식과 낙찰 절차로 계약하나요?
경쟁 방식: 제한경쟁입찰  
낙찰 절차(사업자 선정 방식): 협상에 의한 계약(협상 통해 계약자 선정)

근거: 그랜드코리아레저(주)_2024년도 GKL  그룹웨어 시스템 구축 용역.hwp

[Chroma][supplemental-qa-c05][single_doc] 한국농어촌공사가 네팔에서 물 관리 정보를 디지털화하기 위해 추진하는 시범 시스템은 착수 후 얼마 동안 수행하나요?
용역착수일로부터 6개월입니다. 근거: 한국농어촌공사_네팔 수자원관리 정보화사업-Pilot 시스템 구축용역.hwp

[Chroma][supplemental-qa-c06][si

In [39]:
from src.generation.generation import (
    check_required_facts,
    compute_citation_coverage,
    extract_cited_doc_ids,
    is_abstention,
    compute_abstention_match,
)

In [40]:
print(rag56[0].keys())
print(rag56[0]['gold'].keys())

dict_keys(['absence_scope_doc_ids', 'case_id', 'difficulty', 'enabled', 'evidence_refs', 'gold', 'lane', 'legacy_evidence_note', 'legacy_id', 'legacy_scoring_notes', 'profile', 'question', 'required_doc_ids', 'review', 'reviewed_draft_sha256', 'schema_version', 'scope_doc_ids', 'source_labels', 'source_manifest_sha256', 'source_sha256s', 'supporting_refs', 'supporting_sources', 'tags', 'task_type'])
dict_keys(['abstain_reason', 'comparison_axes', 'decision', 'reference_answer', 'required_fact_groups'])


In [41]:
# 56문항 채점 (팀원 채점 함수 사용)

def score_answers(answers_list, golden_items):
    results = []
    for a in answers_list:
        item = next(it for it in golden_items if it['case_id'] == a['case_id'])
        gold = item['gold']
        answer_text = a['answer']

        # 1) required_fact_groups 매칭
        matched, total = check_required_facts(answer_text, gold.get('required_fact_groups'))
        fact_score = round(matched / total * 100, 2) if total else None

        # 2) 인용 커버리지
        cited_matched, cited_total = compute_citation_coverage(answer_text, item.get('required_doc_ids'))

        # 3) 기권 일치 여부
        expected_abstain = gold.get('decision') in ('abstain', 'source_conflict')
        abstain_match = compute_abstention_match(answer_text, expected_abstain)

        results.append({
            'case_id': a['case_id'],
            'task_type': a['task_type'],
            'fact_score': fact_score,
            'fact_matched': matched,
            'fact_total': total,
            'citation_matched': cited_matched,
            'citation_total': cited_total,
            'abstain_match': abstain_match,
        })
    return results

faiss_results = score_answers(faiss_answers, rag56)
chroma_results = score_answers(chroma_answers, rag56)

In [43]:
import statistics

def summarize(results, label):
    fact_scores = [r['fact_score'] for r in results if r['fact_score'] is not None]
    abstain_matches = [r['abstain_match'] for r in results]
    citation_rates = [(r['citation_matched'], r['citation_total']) for r in results if r['citation_total'] > 0]

    print(f"{label}")
    print(f"평균 fact_score: {statistics.mean(fact_scores):.2f} ({len(fact_scores)}건)")
    print(f"기권 판단 일치율: {sum(abstain_matches)/len(abstain_matches)*100:.2f}% ({len(abstain_matches)}건)")
    total_cited = sum(m for m, t in citation_rates)
    total_required = sum(t for m, t in citation_rates)
    if total_required:
        print(f"인용 커버리지: {total_cited/total_required*100:.2f}% ({total_cited}/{total_required})")
    print()

summarize(faiss_results, "FAISS")
summarize(chroma_results, "Chroma")

FAISS
평균 fact_score: 65.43 (54건)
기권 판단 일치율: 94.64% (56건)
인용 커버리지: 0.00% (0/62)

Chroma
평균 fact_score: 60.96 (54건)
기권 판단 일치율: 94.64% (56건)
인용 커버리지: 0.00% (0/62)



In [44]:
print(faiss_answers[0]['answer'])

￦999,494,600원(부가세 포함)입니다.

근거: 경기도 평택시_2024년도 평택시 버스정보시스템(BIS) 구축사업.hwp


In [45]:
SYSTEM_PROMPT_NEW_V2 += """

기권하는 경우가 아니라면, 답변의 마지막 줄에 실제로 근거로 사용한 출처를
'[근거: doc_id1, doc_id2]' 형식으로 표시해. 근거 헤더의 [문서: doc_id]에
있는 문서명을 그대로 사용해. 사용하지 않은 출처는 포함하지 마.
"""

In [46]:
faiss_answers = []
for item in rag56:
    cid = item['case_id']
    task_type = item['task_type']
    question = item['question']

    answer = ask_rfp_final_faiss(question)
    faiss_answers.append({'case_id': cid, 'task_type': task_type, 'question': question, 'answer': answer})
    print(f"[FAISS][{cid}][{task_type}] {question}")
    print(answer)
    print()

[FAISS][supplemental-qa-c01][single_doc] 평택시가 정류장 이용객에게 실시간 운행정보를 제공하려고 추진한 2024년 사업에는 예산이 얼마나 배정됐나요?
999,494,600원(부가세 포함)

[근거: 경기도 평택시_2024년도 평택시 버스정보시스템(BIS) 구축사업.hwp]

[FAISS][supplemental-qa-c02][single_doc] 평택시의 실시간 버스 도착정보 제공 체계를 새로 만드는 2024년 사업은 착수 후 언제까지 완료해야 하나요?
착수일로부터 2024. 10. 31.까지 완료해야 합니다.

[근거: 경기도 평택시_2024년도 평택시 버스정보시스템(BIS) 구축사업.hwp]

[FAISS][supplemental-qa-c03][single_doc] GKL이 임직원 협업과 전자결재 환경을 새로 구축하려는 2024년 정보화 사업에 책정한 금액은 얼마인가요?
1,515,000,000원 (부가세 포함).  
[근거: 그랜드코리아레저(주)_2024년도 GKL  그룹웨어 시스템 구축 용역.hwp]

[FAISS][supplemental-qa-c04][single_doc] GKL이 사내 협업 업무 환경을 새로 만드는 사업은 어떤 경쟁 방식과 낙찰 절차로 계약하나요?
경쟁 방식: 제한경쟁입찰
낙찰·선정 절차: 협상에 의한 계약(사업자 선정방식) — 평가방식은 기술평가 90%, 가격평가 10%.

[근거: 그랜드코리아레저(주)_2024년도 GKL  그룹웨어 시스템 구축 용역.hwp]

[FAISS][supplemental-qa-c05][single_doc] 한국농어촌공사가 네팔에서 물 관리 정보를 디지털화하기 위해 추진하는 시범 시스템은 착수 후 얼마 동안 수행하나요?
용역착수일로부터 6개월 수행합니다.

[근거: 한국농어촌공사_네팔 수자원관리 정보화사업-Pilot 시스템 구축용역.hwp]

[FAISS][supplemental-qa-c06][single_doc] 한국농어촌공사가 네팔의 수자원 정보를 관리할 시범 시스템을 

In [47]:
chroma_answers = []
for item in rag56:
    cid = item['case_id']
    task_type = item['task_type']
    question = item['question']

    answer = ask_rfp_final_chroma(question)
    chroma_answers.append({'case_id': cid, 'task_type': task_type, 'question': question, 'answer': answer})
    print(f"[Chroma][{cid}][{task_type}] {question}")
    print(answer)
    print()

[Chroma][supplemental-qa-c01][single_doc] 평택시가 정류장 이용객에게 실시간 운행정보를 제공하려고 추진한 2024년 사업에는 예산이 얼마나 배정됐나요?
￦999,494,600원(부가세 포함)

[근거: 경기도 평택시_2024년도 평택시 버스정보시스템(BIS) 구축사업.hwp]

[Chroma][supplemental-qa-c02][single_doc] 평택시의 실시간 버스 도착정보 제공 체계를 새로 만드는 2024년 사업은 착수 후 언제까지 완료해야 하나요?
착수일로부터 2024년 10월 31일까지 완료해야 합니다.

[근거: 경기도 평택시_2024년도 평택시 버스정보시스템(BIS) 구축사업.hwp]

[Chroma][supplemental-qa-c03][single_doc] GKL이 임직원 협업과 전자결재 환경을 새로 구축하려는 2024년 정보화 사업에 책정한 금액은 얼마인가요?
1,515,000천원 (부가세 포함).  
[근거: 그랜드코리아레저(주)_2024년도 GKL  그룹웨어 시스템 구축 용역.hwp]

[Chroma][supplemental-qa-c04][single_doc] GKL이 사내 협업 업무 환경을 새로 만드는 사업은 어떤 경쟁 방식과 낙찰 절차로 계약하나요?
제한경쟁입찰 방식으로 진행되며, 사업자 선정은 협상에 의한 계약(협상에 의해 낙찰자 선정) 방식입니다.  
[근거: 그랜드코리아레저(주)_2024년도 GKL  그룹웨어 시스템 구축 용역.hwp]

[Chroma][supplemental-qa-c05][single_doc] 한국농어촌공사가 네팔에서 물 관리 정보를 디지털화하기 위해 추진하는 시범 시스템은 착수 후 얼마 동안 수행하나요?
용역착수일로부터 6개월간 수행합니다.

[근거: 한국농어촌공사_네팔 수자원관리 정보화사업-Pilot 시스템 구축용역.hwp]

[Chroma][supplemental-qa-c06][single_doc] 한국농어촌공사가 네팔의 수자원 정보를 관리할 시범 시스템을 만드는 데 편성

In [48]:
def score_answers(answers_list, golden_items):
    results = []
    for a in answers_list:
        item = next(it for it in golden_items if it['case_id'] == a['case_id'])
        gold = item['gold']
        answer_text = a['answer']

        matched, total = check_required_facts(answer_text, gold.get('required_fact_groups'))
        fact_score = round(matched / total * 100, 2) if total else None

        cited_matched, cited_total = compute_citation_coverage(answer_text, item.get('required_doc_ids'))

        expected_abstain = gold.get('decision') in ('abstain', 'source_conflict')
        abstain_match = compute_abstention_match(answer_text, expected_abstain)

        results.append({
            'case_id': a['case_id'], 'task_type': a['task_type'],
            'fact_score': fact_score, 'fact_matched': matched, 'fact_total': total,
            'citation_matched': cited_matched, 'citation_total': cited_total,
            'abstain_match': abstain_match,
        })
    return results

faiss_results = score_answers(faiss_answers, rag56)
chroma_results = score_answers(chroma_answers, rag56)

In [49]:
def summarize(results, label):
    fact_scores = [r['fact_score'] for r in results if r['fact_score'] is not None]
    abstain_matches = [r['abstain_match'] for r in results]
    citation_rates = [(r['citation_matched'], r['citation_total']) for r in results if r['citation_total'] > 0]

    print(f"{label}")
    print(f"평균 fact_score: {statistics.mean(fact_scores):.2f} ({len(fact_scores)}건)")
    print(f"기권 판단 일치율: {sum(abstain_matches)/len(abstain_matches)*100:.2f}% ({len(abstain_matches)}건)")
    total_cited = sum(m for m, t in citation_rates)
    total_required = sum(t for m, t in citation_rates)
    if total_required:
        print(f"인용 커버리지: {total_cited/total_required*100:.2f}% ({total_cited}/{total_required})")
    else:
        print("인용 커버리지: 계산 불가 (required_doc_ids 없음)")
    print()

summarize(faiss_results, "FAISS")
summarize(chroma_results, "Chroma")

FAISS
평균 fact_score: 62.19 (54건)
기권 판단 일치율: 94.64% (56건)
인용 커버리지: 0.00% (0/62)

Chroma
평균 fact_score: 63.18 (54건)
기권 판단 일치율: 94.64% (56건)
인용 커버리지: 0.00% (0/62)



In [50]:
print(faiss_answers[0]['answer'])

999,494,600원(부가세 포함)

[근거: 경기도 평택시_2024년도 평택시 버스정보시스템(BIS) 구축사업.hwp]


In [52]:
print(rag56[0].get('source_labels'))
print(rag56[0].get('required_doc_ids'))

['경기도 평택시_2024년도 평택시 버스정보시스템(BIS) 구축사업.hwp']
['doc_67e38333d2e31eb4543e02d9']


In [53]:
from src.evaluation.golden_set_v3 import _normalize_filename

def compute_citation_coverage_by_label(answer, source_labels):
    """required_doc_ids(해시) 대신 source_labels(실제 파일명)로 인용 커버리지 계산"""
    if not source_labels:
        return 0, 0
    cited = extract_cited_doc_ids(answer)
    cited_norm = {_normalize_filename(c) for c in cited}
    required_norm = [_normalize_filename(lb) for lb in source_labels]
    matched = sum(1 for r in required_norm if r in cited_norm)
    return matched, len(required_norm)

In [54]:
def score_answers(answers_list, golden_items):
    results = []
    for a in answers_list:
        item = next(it for it in golden_items if it['case_id'] == a['case_id'])
        gold = item['gold']
        answer_text = a['answer']

        matched, total = check_required_facts(answer_text, gold.get('required_fact_groups'))
        fact_score = round(matched / total * 100, 2) if total else None

        cited_matched, cited_total = compute_citation_coverage_by_label(answer_text, item.get('source_labels'))

        expected_abstain = gold.get('decision') in ('abstain', 'source_conflict')
        abstain_match = compute_abstention_match(answer_text, expected_abstain)

        results.append({
            'case_id': a['case_id'], 'task_type': a['task_type'],
            'fact_score': fact_score, 'fact_matched': matched, 'fact_total': total,
            'citation_matched': cited_matched, 'citation_total': cited_total,
            'abstain_match': abstain_match,
        })
    return results

faiss_results = score_answers(faiss_answers, rag56)
chroma_results = score_answers(chroma_answers, rag56)

summarize(faiss_results, "FAISS")
summarize(chroma_results, "Chroma")

FAISS
평균 fact_score: 62.19 (54건)
기권 판단 일치율: 94.64% (56건)
인용 커버리지: 84.38% (54/64)

Chroma
평균 fact_score: 63.18 (54건)
기권 판단 일치율: 94.64% (56건)
인용 커버리지: 81.25% (52/64)



In [55]:
with open(DATA_DIR / 'dev.refined.review-candidate.jsonl', encoding='utf-8') as f:
    core40 = [json.loads(l) for l in f if l.strip()]

print(f"총 문항 수: {len(core40)}")
print(core40[0].keys())

총 문항 수: 40
dict_keys(['case_id', 'conversation', 'difficulty', 'document_scope', 'gold', 'group_id', 'history', 'question', 'review', 'schema_version', 'source_manifest_sha256', 'split', 'tags', 'task_type'])


In [56]:
print(core40[0]['gold'].keys())
print(core40[0]['document_scope'])

# follow_up 유형 하나 확인
followup_item = next(it for it in core40 if it['task_type'] == 'follow_up')
print(followup_item['history'])

dict_keys(['abstain_reason', 'comparison_axes', 'decision', 'evidence_refs', 'reference_answer', 'required_doc_ids', 'required_key_points'])
{'doc_ids': ['doc_e3b910313338c8c5232ec2de'], 'mode': 'explicit'}
[{'content': 'BIFF&ACFM 사업 예산이 얼마야?', 'role': 'user', 'turn_id': 'fu001.u1'}, {'cited_doc_ids': ['doc_e3b910313338c8c5232ec2de'], 'content': 'VAT 포함 243,000,000원입니다.', 'role': 'assistant', 'turn_id': 'fu001.a1'}]


In [57]:
print(core40[0]['gold']['evidence_refs'])

[{'doc_id': 'doc_e3b910313338c8c5232ec2de', 'locator_hash': '999e9ebe39e2c9f50098cb5d1a0ed39451f8ec8642ed7d5c11d14c6e9d3bd2ac', 'source_block_id': 'block_bf5e8dd0fb826bd4e798c317'}]


In [58]:
def score_core40(answers_list, golden_items):
    results = []
    for a in answers_list:
        item = next(it for it in golden_items if it['case_id'] == a['case_id'])
        gold = item['gold']
        answer_text = a['answer']

        matched, total = check_required_facts(answer_text, gold.get('required_key_points'))
        fact_score = round(matched / total * 100, 2) if total else None

        expected_abstain = gold.get('decision') in ('abstain', 'source_conflict')
        abstain_match = compute_abstention_match(answer_text, expected_abstain)

        results.append({
            'case_id': a['case_id'], 'task_type': a['task_type'],
            'fact_score': fact_score, 'fact_matched': matched, 'fact_total': total,
            'abstain_match': abstain_match,
        })
    return results

In [59]:
print(core40[0]['gold']['required_key_points'])

[{'point_id': 'kp_01', 'text': '사업예산은 243,000,000원이다.'}, {'point_id': 'kp_02', 'text': '예산에는 VAT가 포함된다.'}]


In [61]:
def convert_key_points_to_fact_groups(required_key_points):
    if not required_key_points:
        return []
    return [[kp['text']] for kp in required_key_points]

def score_core40(answers_list, golden_items):
    results = []
    for a in answers_list:
        item = next(it for it in golden_items if it['case_id'] == a['case_id'])
        gold = item['gold']
        answer_text = a['answer']

        fact_groups = convert_key_points_to_fact_groups(gold.get('required_key_points'))
        matched, total = check_required_facts(answer_text, fact_groups)
        fact_score = round(matched / total * 100, 2) if total else None

        expected_abstain = gold.get('decision') in ('abstain', 'source_conflict')
        abstain_match = compute_abstention_match(answer_text, expected_abstain)

        results.append({
            'case_id': a['case_id'], 'task_type': a['task_type'],
            'fact_score': fact_score, 'fact_matched': matched, 'fact_total': total,
            'abstain_match': abstain_match,
        })
    return results

In [62]:
faiss_answers_40 = []
for item in core40:
    cid = item['case_id']
    task_type = item['task_type']
    question = item['question']

    if task_type == 'follow_up':
        history = item.get('history', [])
        user_turns = [h['content'] for h in history if h.get('role') == 'user']
        prev_q = user_turns[-1] if user_turns else ""
        combined_q = f"{prev_q} {question}"
        answer = ask_rfp_final_faiss(combined_q)
    else:
        answer = ask_rfp_final_faiss(question)

    faiss_answers_40.append({'case_id': cid, 'task_type': task_type, 'answer': answer})
    print(f"[FAISS][{cid}][{task_type}] {question}")
    print(answer)
    print()

[FAISS][dev-single-001][single_doc] BIFF&ACFM 온라인서비스 재개발 사업의 사업예산은 얼마이며 부가가치세가 포함돼 있나요?
사업예산: 금 243,000,000원 — 부가가치세(VAT) 포함입니다.

[근거: (사)부산국제영화제_2024년 BIFF & ACFM 온라인서비스 재개발 및 행사지원시.hwp]

[FAISS][dev-single-002][single_doc] e나라도움 웹 접근성 컨설팅의 수행기간과 예산을 알려줘.
- 수행기간: 계약일로부터 120일(약 4개월)  
- 예산: 70,000,000원(금칠천만원, VAT 포함)

[근거: 한국재정정보원_e나라도움 업무시스템 웹 접근성 컨설팅.hwp]

[FAISS][dev-single-003][single_doc] 국립인천해양박물관 해양자료관리시스템 사업은 몇 차로 나뉘고 기술평가와 가격평가 비중은 어떻게 되나요?
- 사업은 2차(1차: 계약일로부터 4개월, 2차: 1차 완료일로부터 2개월)로 나뉩니다.
- 평가 비중은 기술평가 90%, 가격평가 10%입니다.

[근거: 국립인천해양박물관_국립인천해양박물관 해양자료관리시스템 구축 용.hwp]

[FAISS][dev-single-004][single_doc] RFID 기반 국가자격 시험 결과물 관리시스템의 사업기간과 시범 도입 규모는?
사업기간: 계약일로부터 90일 (사업기간 표기).  
시범 도입 규모: 제공된 문서 범위에서는 명시된 시범 도입 규모가 확인되지 않습니다. (납품 장소로 서울지역본부·울산지사·국가자격채점센터가 기재되어 있으나, 이를 '시범 도입 규모'로 확정할 근거는 문서에 없습니다.)

[근거: 한국산업인력공단_RFID기반 국가자격 시험 결과물 스마트 관리시스템 도.hwp]

[FAISS][dev-single-005][single_doc] 서민금융 채팅 상담시스템 입찰에서 입찰서와 제안서는 어떻게 제출하고 파일 형식·용량 제한은 무엇인가요?
- 제출 방법: 입찰서와 제안서는 전자적으로 제출해야 하며(나라장터를

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

[FAISS][dev-unknown-001][unknown] 오늘 새로 올라온 나라장터 공고 세 개만 알려줘.
실시간·오늘 올라온 공고 조회는 제공된 문서에서 확인할 수 없습니다. 이 기능은 제가 수행할 수 없습니다.

[FAISS][dev-unknown-002][unknown] BIFF&ACFM 온라인서비스 재개발 사업의 최종 낙찰업체와 낙찰금액은 얼마야?
확인되지 않습니다.

[근거: (사)부산국제영화제_2024년 BIFF & ACFM 온라인서비스 재개발 및 행사지원시.hwp]



Batches:   0%|          | 0/1 [00:00<?, ?it/s]

[FAISS][dev-unknown-003][unknown] 공고 담당자에게 지금 전화해서 제안설명회 참석한다고 전해줘.
이 기능은 제가 수행할 수 없습니다.

[FAISS][dev-unknown-004][unknown] 우리 회사가 BIFF&ACFM 사업의 입찰참가자격을 모두 충족하는지 판정해줘.
답변할 수 없습니다.

문서만으로 귀사가 실제로 자격을 충족하는지 판정하는 것은 허용되지 않습니다. 대신 귀사가 스스로 대조해서 판정할 수 있도록, 문서에 명시된 입찰참가자격 항목들을 간단히 정리해드립니다. 아래 항목을 귀사 자료와 하나씩 대조해 주세요. 일부 항목은 증빙서류(예: 직접생산확인증명서, 나라장터 등록 등)가 필요합니다.

필수 확인 항목(문서 기준)
- 부정당업자 해당여부: 「지방자치단체를 당사자로 하는 계약에 관한 법령」 제92조에 해당하지 않아야 함.
- 주된 영업소 소재지: 입찰공고일 전일부터 계약체결일까지 주된 영업소가 부산광역시 소재여야 함(동법 시행규칙 근거).
- 나라장터(G2B) 등록: 입찰서 제출마감일 전까지 소프트웨어사업자(업종코드 1468, 컴퓨터관련서비스사업)로 입찰참가자격 등록 완료.
- 기업 규모 제한: 소프트웨어산업 진흥법 및 관련 지침에 따라 대기업·중견기업, 상호출자제한기업집단 소속회사는 참여 불가(중소 소프트웨어 사업자여야 함).
- 직접생산확인증명서: 정보시스템개발서비스(세부품명번호 8111159901)에 대한 ‘직접생산확인증명서’ 보유(입찰마감 전까지 발급·유효기간 내).
- 공동수급·하도급: 공동수급(공동이행방식) 및 하도급은 허용되지 않음(단독 참여여부 확인).

제출·평가 관련 유의사항(증빙 필요 가능 항목)
- 산출내역: 용역비는 직접인건비·기술료·재경비·공급비용·부가세 등으로 구분해 산정해야 함.
- S/W 개발에 필요한 작업장·장비는 제안사가 제공해야 하며 임대비 별도 산정 불가.

위 항목 중 귀사에서 확인되지 않는 항목이 있으면 “확인되지 않습니다” 상태로 판단되어 원문 전체(또는 관련 증빙) 확인이 

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

[FAISS][dev-unknown-007][unknown] 평가위원들이 실제로 더 중요하게 보는 비공개 평가 포인트를 알려줘.
답변할 수 없습니다. 문서에 없는 ‘평가위원들의 실제 비공개(내부) 우선순위’를 확정해서 알려달라는 요청은 문서 범위를 벗어난 판단/추측에 해당합니다. 문서에 없는 정보를 추정·제시할 수 없습니다.

대신, 제공된 문서들에서 명시되어 있어 평가 시 중요하게 다루어질 수 있다고 근거로 확인되는 공개된 평가항목·유의사항을 정리해 드립니다(이 항목들은 문서상 공개된 기준·강조점이며, 개별 위원의 비공개 우선순위와 동일하다고 단정할 수는 없습니다).

- 제안서 표현의 명확성: 모호한 표현(“~할 수도 있다”, “~이 가능하다” 등)은 불리하게 평가됨 — 제안서의 문구 정확성 및 확답성 중요.  
  근거: [문서: 수협중앙회_강릉어선안전조업국 상황관제시스템 구축.hwp]

- 기술·기능·성능·품질 관련 항목: 요구사항에 대한 적정성, 성능확보방안, 품질보증방안 등 기술적 완결성·실현가능성 중시.  
  근거: [문서: 서울특별시_2024년 지도정보 플랫폼 및 전문활용 연계 시스템 고도화 용.pdf]

- 프로젝트 관리·지원: 일정·진도·위험관리·형상관리 등 관리방안, 교육·기술이전·하자보수·비상대책·기밀보안 등 운영·지원 계획의 충실성.  
  근거: [문서: 서울특별시_2024년 지도정보 플랫폼 및 전문활용 연계 시스템 고도화 용.pdf]

- 수행실적·유사사업 실적 및 경영상태: 최근 유사 사업의 실적(금액·건수), 기업신용등급, 인증 등 객관적 실적·신뢰성 요소 반영.  
  근거: [문서: 서울특별시_2024년 지도정보 플랫폼 및 전문활용 연계 시스템 고도화 용.pdf]

- 정성적 평가(전략·방법론, 창의성·타당성 등): 사업이해도, 추진전략·방법론의 타당성·확장성 등 정성평가 요소 비중 존재.  
  근거: [문서: 서울특별시_2024년 지도정보 플랫폼 및 전문활용 연계 시스템 고도화 용.pdf]

- 평가점수 공개·비공개 관련

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

[FAISS][dev-unknown-009][unknown] 입찰 참여 시작일이 문서에 없으면 공개일 다음 날이라고 보고 날짜를 확정해줘.
확인되지 않습니다.

입찰 참여 시작일은 제공된 문서에 명확히 명시되어야 하며, 문서에 없는 값을 공개일 다음 날로 임의 확정할 수 없습니다. (제공된 문서 범위에서는 확인되지 않습니다.)

[FAISS][dev-unknown-010][unknown] BIFF&ACFM 온라인서비스 재개발 사업은 현재 몇 퍼센트 진행됐고 계약 지연 가능성은 얼마나 돼?
답변할 수 없습니다.

문서(제안요청서)만으로는 “현재 진행률(%)”과 “계약 지연 가능성(확률·정성적 판단)”을 판단할 근거가 없습니다. 이런 판단에는 실제 진행상황과 진척보고서, 변경내역, 일정상 문제(인력·결제·승인 지연 등)에 대한 최신 정보가 필요합니다.

대신 컨설턴트가 판단하려면 확인해야 할 최소 항목(문서)을 간단히 제시합니다.
- 현재 완료된 산출물 목록 및 산출물별 완료일(진척보고서, 완료검수 기록)
- 원래 계약·착수 일정 및 최신 변경된 일정(공정표, 일정변경 통지)
- 납품·검수·승인 현황(검수결과서, 수용/반려 이력)
- 계약 변경(추가과업·범위 변경·요금조정) 및 발주기관·수행사 간 서신
- 투입인력 및 장비 현황(파견인력 출근기록, 리소스 할당)
- 품목별 이슈·리스크 로그(지연 사유 기록, 대응계획)
- 결제·예산 지출 현황(대금지급 일정·미지급 내역)

위 자료들을 제공해주시면, 문서 범위 내에서 확인 가능한 근거로 진행률 산정에 필요한 항목을 도출하거나 지연 리스크 평가에 필요한 확인사항을 정리해 드리겠습니다.

[근거: (사)부산국제영화제_2024년 BIFF & ACFM 온라인서비스 재개발 및 행사지원시.hwp]



In [63]:
chroma_answers_40 = []
for item in core40:
    cid = item['case_id']
    task_type = item['task_type']
    question = item['question']

    if task_type == 'follow_up':
        history = item.get('history', [])
        user_turns = [h['content'] for h in history if h.get('role') == 'user']
        prev_q = user_turns[-1] if user_turns else ""
        combined_q = f"{prev_q} {question}"
        answer = ask_rfp_final_chroma(combined_q)
    else:
        answer = ask_rfp_final_chroma(question)

    chroma_answers_40.append({'case_id': cid, 'task_type': task_type, 'answer': answer})
    print(f"[Chroma][{cid}][{task_type}] {question}")
    print(answer)
    print()

[Chroma][dev-single-001][single_doc] BIFF&ACFM 온라인서비스 재개발 사업의 사업예산은 얼마이며 부가가치세가 포함돼 있나요?
사업예산: 243,000,000원 — 부가가치세(VAT) 포함.  
[근거: (사)부산국제영화제_2024년 BIFF & ACFM 온라인서비스 재개발 및 행사지원시.hwp]

[Chroma][dev-single-002][single_doc] e나라도움 웹 접근성 컨설팅의 수행기간과 예산을 알려줘.
수행기간: 계약일로부터 120일(약 4개월)  
예산: 70,000,000원(부가세 포함)

[근거: 한국재정정보원_e나라도움 업무시스템 웹 접근성 컨설팅.hwp]

[Chroma][dev-single-003][single_doc] 국립인천해양박물관 해양자료관리시스템 사업은 몇 차로 나뉘고 기술평가와 가격평가 비중은 어떻게 되나요?
- 사업은 2차로 구분됩니다. (1차: 계약일로부터 4개월, 2차: 1차 완료일로부터 2개월)

- 평가 비중은 기술평가 90% / 가격평가 10%입니다.

[근거: 국립인천해양박물관_국립인천해양박물관 해양자료관리시스템 구축 용.hwp]

[Chroma][dev-single-004][single_doc] RFID 기반 국가자격 시험 결과물 관리시스템의 사업기간과 시범 도입 규모는?
사업기간: 계약일로부터 90일(문서 내 추진일정에서는 계약일로부터 3개월로 표기됨).  
시범 도입 규모: 제공된 문서 범위에서는 시범 도입의 구체적 규모(장비 대수, 적용 장소 수 등)는 확인되지 않습니다.

[근거: 한국산업인력공단_RFID기반 국가자격 시험 결과물 스마트 관리시스템 도.hwp]

[Chroma][dev-single-005][single_doc] 서민금융 채팅 상담시스템 입찰에서 입찰서와 제안서는 어떻게 제출하고 파일 형식·용량 제한은 무엇인가요?
- 제출 방법: 입찰서와 제안서는 전자입찰(나라장터)을 통해 전자적으로만 제출해야 하며, 입찰서와 제안서를 모두 제출해야 유효합니다. (입찰서

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

[Chroma][dev-unknown-001][unknown] 오늘 새로 올라온 나라장터 공고 세 개만 알려줘.
실시간·오늘 기준으로 올라온 나라장터 공고 조회는 제공된 문서만으로 확인할 수 없습니다. 이 기능은 제가 수행할 수 없습니다.

[Chroma][dev-unknown-002][unknown] BIFF&ACFM 온라인서비스 재개발 사업의 최종 낙찰업체와 낙찰금액은 얼마야?
확인되지 않습니다.

[근거: (사)부산국제영화제_2024년 BIFF & ACFM 온라인서비스 재개발 및 행사지원시.hwp]



Batches:   0%|          | 0/1 [00:00<?, ?it/s]

[Chroma][dev-unknown-003][unknown] 공고 담당자에게 지금 전화해서 제안설명회 참석한다고 전해줘.
이 기능은 제가 수행할 수 없습니다. 답변할 수 없습니다.

[근거: 한국수자원조사기술원_수문자료정보관리시스템(HDIMS) 재구축 용역(3단계.hwp)]

[Chroma][dev-unknown-004][unknown] 우리 회사가 BIFF&ACFM 사업의 입찰참가자격을 모두 충족하는지 판정해줘.
문서만으로 귀사가 자격을 충족하는지 판정할 수 없습니다.

대신 판정에 필요한 확인항목(문서에 명시된 입찰참가자격)을 정리합니다. 각 항목을 귀사 정보와 대조해 모두 '예'이면 자격을 충족할 가능성이 있습니다(최종 판단은 제출서류로 확인되어야 함).

필수 확인항목:
1. 부정당업자 해당 여부: 지방자치단체를 당사자로 하는 계약에 관한 법령(시행령 제92조)상의 부정당업자에 해당하지 않을 것.  
2. 주된 영업소 소재지: 입찰공고일 전일부터 계약체결일까지 주된 영업소 소재지가 부산광역시일 것(지방자치단체 법령·시행규칙 근거).  
3. 나라장터(G2B) 등록: 입찰서 제출마감일 전일까지 나라장터에 소프트웨어사업자(업종코드 1468, 컴퓨터관련서비스사업)로 입찰참가자격 등록 완료 여부.  
4. 기업규모 제한: 소프트웨어산업 진흥법·관련 지침에 따라 대기업·중견기업 및 상호출자제한기업집단 소속회사가 아님(참여 불가).  
5. 직접생산확인증명서: 정보시스템개발서비스(세부품명번호 8111159901)에 대한 '직접생산확인증명서'를 입찰마감 전일까지 발급·유효기간 내에 소지할 것.  
6. 공동수급·하도급 제한: 공동수급(공동이행방식) 및 하도급을 허용하지 않으므로 단독으로 입찰·이행 가능한지(하도급 포함 불가) 확인.  
7. 기타 제출·준수사항: 제안서·증빙서류의 성실 제출, 청렴계약 등 관련 서약·조건 준수 가능 여부.

위 항목 각각에 대해 귀사 내부 자료(사업자등록증, 본사·주된 영업소 주소, 나라장터 등록현황, 직접생산확인증명서, 기업집단 

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

[Chroma][dev-unknown-007][unknown] 평가위원들이 실제로 더 중요하게 보는 비공개 평가 포인트를 알려줘.
요청하신 종류의 판단(평가위원들이 '실제 더 중요하게 보는 비공개 평가 포인트'를 추정·단정)은 제공된 문서 범위를 벗어난 주관적 추정에 해당하므로 답변할 수 없습니다.

대신, 제공된 문서들에 명시된 평가항목·배점·평가절차 등 명확한 근거를 기반으로 한 요약(문서별 공개된 평가기준과 발표·질의응답 규칙, 정성/정량 항목 등)을 제공해 드릴 수 있습니다. 원하시면 다음 중 선택해 주세요.
- 문서별 공개된 평가항목·배점(간단 요약)
- 발표·질의응답 시 유의사항 및 제안서 작성 시 문서상 강조된 금지표현/증빙요건 정리
- 여러 문서를 비교해 반복적으로 등장하는 공개 평가항목 정리(추세 파악 수준, 추측 아님)

원하시는 항목을 알려주시면 제공된 문서 내용만으로 요약해 드리겠습니다.  
[근거: 수협중앙회_강릉어선안전조업국 상황관제시스템 구축.hwp, 한국생산기술연구원_EIP3.0 고압가스 안전관리 시스템 구축 용역.hwp, 부산관광공사_경영정보시스템 기능개선.hwp]

[Chroma][dev-unknown-008][unknown] BIFF&ACFM 사업과 서민금융 채팅상담 사업 중 우리 회사 수주확률이 더 높은 곳을 퍼센트로 계산해줘.
이 질문은 제안문서만으로 수주확률(퍼센트)을 계산할 수 있는 판단·추측 요청에 해당하므로 답변할 수 없습니다.

대신 수주 가능성 판단을 위해 필요한 핵심 정보(문서에 없으면 외부 확인 필요)를 간단히 안내합니다.
- 당사 과거 실적(유사사업 수주·수행 이력) 및 레퍼런스
- 제안서(기술·가격) 경쟁력 비교 자료(타사 예상 경쟁력 포함)
- 회사의 인력·조직·재무능력(파견 가능 인력 포함)
- 입찰조건(지역제한, 긴급공고 여부, 분리발주·직접구매 영향 등)과 해당 사업의 평가기준·배점
- 경쟁사 수(응찰 예상 업체 수) 및 주요 경쟁사 프로필
- 입찰 제한사항·자격 요건 충족 여부(제안요청서에 명시된

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

[Chroma][dev-unknown-009][unknown] 입찰 참여 시작일이 문서에 없으면 공개일 다음 날이라고 보고 날짜를 확정해줘.
문서에 명시되지 않은 내용을 사용자가 제시한 규칙(공개일 다음 날로 간주)으로 확정해 달라는 요청은 임의 가정에 따른 확정 답변을 요구하는 상황입니다. 제공된 문서에 근거하지 않은 임의 가정으로 날짜를 확정할 수 없으므로 답변할 수 없습니다.

원하시면 다음 중 하나를 선택해 주세요.
- 문서에서 ‘공개일’(공고일 등)이 명시된 개별 문서를 지정해 주시면, 그 문서의 공개일을 근거로 사용자가 제시한 규칙을 적용한 가정 날짜를 산출해 보여 드리겠습니다(단, 이는 사용자의 가정에 따른 산출임을 명확히 표기합니다).
- 또는 제가 제공된 문서들에서 ‘공개일/공고일’ 표기가 있는 문서를 찾아드릴게요. 이후 사용자가 가정 적용 여부를 확정해 주세요.

[근거: 중앙선거관리위원회_2025년도 행정정보시스템 위탁운영사업.hwp, 수협중앙회_수협중앙회 수산물사이버직매장 시스템 재구축 ISMP 수립 입.hwp, 한국농어촌공사_아세안+3 식량안보정보시스템(AFSIS) 3단계 협력(캄보디아.hwp, 경희대학교_[입찰공고] 산학협력단 정보시스템 운영 용역업체 선정.hwp, 사단법인아시아물위원회사무국_우즈벡-키르기즈스탄 기후변화대응 스.hwp, 한국수출입은행_(긴급) 모잠비크 마푸토 지능형교통시스템(ITS) 구축사업.hwp, 인천광역시 동구_수도국산달동네박물관 전시해설 시스템 구축(협상에 .hwp, 한국원자력연구원_한국원자력연구원 선량평가시스템 고도화.hwp, 한국보건산업진흥원_의료기기산업 종합정보시스템(정보관리기관) 기능.hwp, 한국연구재단_2024년 기초학문자료센터 시스템 운영 및 연구성과물 DB구.hwp]

[Chroma][dev-unknown-010][unknown] BIFF&ACFM 온라인서비스 재개발 사업은 현재 몇 퍼센트 진행됐고 계약 지연 가능성은 얼마나 돼?
답변할 수 없습니다.

제공된 제안요청서(RFP) 문서만으로는 현재 

In [64]:
faiss_results_40 = score_core40(faiss_answers_40, core40)
chroma_results_40 = score_core40(chroma_answers_40, core40)

def summarize_core40(results, label):
    fact_scores = [r['fact_score'] for r in results if r['fact_score'] is not None]
    abstain_matches = [r['abstain_match'] for r in results]

    print(f"{label}")
    print(f"평균 fact_score: {statistics.mean(fact_scores):.2f} ({len(fact_scores)}건)")
    print(f"기권 판단 일치율: {sum(abstain_matches)/len(abstain_matches)*100:.2f}% ({len(abstain_matches)}건)")
    print()

summarize_core40(faiss_results_40, "FAISS")
summarize_core40(chroma_results_40, "Chroma")

FAISS
평균 fact_score: 4.44 (30건)
기권 판단 일치율: 75.00% (40건)

Chroma
평균 fact_score: 4.44 (30건)
기권 판단 일치율: 75.00% (40건)



In [65]:
# 첫 문항으로 원인 확인
item = core40[0]
answer = faiss_answers_40[0]['answer']

print("질문:", item['question'])
print("답변:", answer)
print()
print("required_key_points:", item['gold']['required_key_points'])
print()

fact_groups = convert_key_points_to_fact_groups(item['gold']['required_key_points'])
print("변환된 fact_groups:", fact_groups)
matched, total = check_required_facts(answer, fact_groups)
print(f"매칭: {matched}/{total}")

질문: BIFF&ACFM 온라인서비스 재개발 사업의 사업예산은 얼마이며 부가가치세가 포함돼 있나요?
답변: 사업예산: 금 243,000,000원 — 부가가치세(VAT) 포함입니다.

[근거: (사)부산국제영화제_2024년 BIFF & ACFM 온라인서비스 재개발 및 행사지원시.hwp]

required_key_points: [{'point_id': 'kp_01', 'text': '사업예산은 243,000,000원이다.'}, {'point_id': 'kp_02', 'text': '예산에는 VAT가 포함된다.'}]

변환된 fact_groups: [['사업예산은 243,000,000원이다.'], ['예산에는 VAT가 포함된다.']]
매칭: 0/2


In [66]:
# core40 채점 (네 official_score 방식, 최종본)

def normalize_text(t):
    return t.replace(',', '').replace(' ', '')

def normalize_dates(text):
    text = re.sub(r'(\d{4})년\s*(\d{1,2})월\s*(\d{1,2})일', r'\1.\2.\3', text)
    text = re.sub(r'(\d{4})\.(\d{1,2})\.(\d{1,2})', lambda m: f"{m.group(1)}.{int(m.group(2)):02d}.{int(m.group(3)):02d}", text)
    return text

def _text_included(fact_text, answer_text):
    fact_text = normalize_dates(fact_text)
    answer_text_n = normalize_dates(answer_text)
    answer_norm = normalize_text(answer_text_n)

    numbers = re.findall(r'\d+(?:\.\d+)?', fact_text)
    numbers = [n for n in numbers if len(n) >= 2]
    for num in numbers:
        if num in answer_norm:
            continue
        if len(num) == 4 and num.startswith('20'):
            if num[2:] in answer_norm:
                continue
        num_no_zero = re.sub(r'^0+', '', num)
        if num_no_zero and num_no_zero in answer_norm:
            continue
        return False

    raw_words = re.split(r'[\s,·:()]+', fact_text)
    stopwords_suffix = ('이다', '한다', '된다', '있다', '없다', '이며', '해야', '까지', '입니다', '부터', '년', '월', '일')
    core_words = []
    for w in raw_words:
        w = w.rstrip('.,')
        if len(w) < 2:
            continue
        if re.match(r'^\d', w):
            continue
        for suf in stopwords_suffix:
            if w.endswith(suf) and len(w) > len(suf):
                w = w[:-len(suf)]
                break
        w = re.sub(r'(은|는|이|가|을|를|에|의|와|과|로|으로)$', '', w)
        if len(w) >= 2:
            core_words.append(w)

    if not core_words:
        return True

    match_count = sum(1 for w in core_words if w in answer_text or normalize_text(w) in answer_norm)
    return match_count / max(len(core_words), 1) >= 0.4

ABSTAIN_PHRASES = ['확인되지 않습니다', '답변할 수 없', '수행할 수 없', '확인할 수 없', '판단할 수 없',
                    '판정할 수 없', '판정해줄 수 없', '계산할 수 없', '제공할 수 없', '받아들일 수 없',
                    '확정할 수 없', '알려드릴 수 없', '불가능합니다', '제공된 문서 범위에서는']

def official_score_core40(item, answer_text):
    gold = item['gold']
    decision = gold.get('decision')

    if decision == 'abstain':
        is_abstained = any(p in answer_text for p in ABSTAIN_PHRASES)
        return 100 if is_abstained else 0

    key_points = gold.get('required_key_points', [])
    if not key_points:
        return None

    included = [_text_included(kp['text'], answer_text) for kp in key_points]
    return round(sum(included) / len(included) * 100, 2)

In [67]:
def score_core40_v2(answers_list, golden_items):
    results = []
    for a in answers_list:
        item = next(it for it in golden_items if it['case_id'] == a['case_id'])
        score = official_score_core40(item, a['answer'])
        results.append({'case_id': a['case_id'], 'task_type': a['task_type'], 'score': score})
    return results

faiss_results_40 = score_core40_v2(faiss_answers_40, core40)
chroma_results_40 = score_core40_v2(chroma_answers_40, core40)

def summarize_score(results, label):
    valid = [r['score'] for r in results if r['score'] is not None]
    print(f"{label}")
    print(f"전체 평균: {sum(valid)/len(valid):.2f}/100 ({len(valid)}개)")
    by_type = {}
    for r in results:
        if r['score'] is not None:
            by_type.setdefault(r['task_type'], []).append(r['score'])
    for t, scores in by_type.items():
        print(f"{t}: 평균 {sum(scores)/len(scores):.2f}/100 ({len(scores)}개)")
    print()

summarize_score(faiss_results_40, "FAISS")
summarize_score(chroma_results_40, "Chroma")

FAISS
전체 평균: 88.75/100 (40개)
single_doc: 평균 90.83/100 (10개)
multi_doc_compare: 평균 77.50/100 (10개)
follow_up: 평균 86.67/100 (10개)
unknown: 평균 100.00/100 (10개)

Chroma
전체 평균: 88.96/100 (40개)
single_doc: 평균 89.17/100 (10개)
multi_doc_compare: 평균 80.00/100 (10개)
follow_up: 평균 86.67/100 (10개)
unknown: 평균 100.00/100 (10개)



In [68]:
# 버전 C: Chroma 순수 벡터 검색 (Hybrid 아님, FAISS와 공정 비교용)

def ask_rfp_final_chroma_vector_only(question, model_name="gpt-5-mini", max_retries=2):
    doc_hints = extract_doc_hints_multi(question, all_filenames_with_biz)
    doc_hints = doc_hints[:3]
    keywords = find_relevant_keywords(question)
    conditions = extract_filter_conditions(question)

    doc_to_meta = {}
    for c in child_chunks:
        if c.doc_id not in doc_to_meta:
            doc_to_meta[c.doc_id] = c.metadata

    context_parts = []

    def get_doc_chunks(doc_id):
        return [c for c in child_chunks if c.doc_id == doc_id]

    def build_meta_filter(conds):
        if not conds:
            return None
        def _filter(meta):
            if '금액_최소' in conds:
                amt = meta.get('사업_금액')
                if amt is None or amt < conds['금액_최소']:
                    return False
            if conds.get('지자체'):
                if not is_local_gov(meta.get('발주_기관')):
                    return False
            if conds.get('공사'):
                org = str(meta.get('발주_기관', ''))
                if '공사' not in org:
                    return False
            return True
        return _filter

    if is_aggregation_question(question) and len(doc_hints) >= 1:
        stopwords_q = {'사업의', '사업에서', '어떻게', '되나요?', '되나요', '몇', '어떤', '얼마', '비교'}
        qkeywords = [w for w in re.split(r'[ ,]', question) if len(w) >= 2 and w not in stopwords_q]
        best_doc, best_score = doc_hints[0], -1
        for fname in doc_hints:
            biz = doc_to_meta.get(fname, {}).get('발주_기관', '')
            score = sum(1 for kw in qkeywords if kw in fname or kw in str(biz))
            if score > best_score:
                best_score, best_doc = score, fname
        doc_hint = best_doc
        header = meta_header_from_metadata(doc_hint, doc_to_meta.get(doc_hint, {}))
        for c in get_doc_chunks(doc_hint):
            context_parts.append(f"{header}\n{c.text}")

    elif len(doc_hints) == 1 and keywords:
        doc_hint = doc_hints[0]
        doc_c = get_doc_chunks(doc_hint)
        keyword_chunks = [c for c in doc_c if any(kw in c.text for kw in keywords)]
        header = meta_header_from_metadata(doc_hint, doc_to_meta.get(doc_hint, {}))
        if keyword_chunks:
            for c in keyword_chunks[:25]:
                context_parts.append(f"{header}\n{c.text}")
        else:
            hits = index.vector_search(question, k=10, expand_to_parent=True)  # ← vector_search로 변경
            for h in hits:
                context_parts.append(f"{meta_header_from_metadata(h.doc_id, doc_to_meta.get(h.doc_id, {}))}\n{h.text}")

    elif len(doc_hints) >= 2:
        for doc_hint in doc_hints:
            doc_c = get_doc_chunks(doc_hint)
            if keywords:
                matched = [c for c in doc_c if any(kw in c.text for kw in keywords)]
                selected = matched[:8] if matched else doc_c[:8]
            else:
                selected = doc_c[:8]
            header = meta_header_from_metadata(doc_hint, doc_to_meta.get(doc_hint, {}))
            for c in selected:
                context_parts.append(f"{header}\n{c.text}")

    elif doc_hints:
        doc_hint = doc_hints[0]
        doc_c = get_doc_chunks(doc_hint)
        header = meta_header_from_metadata(doc_hint, doc_to_meta.get(doc_hint, {}))
        for c in doc_c[:15]:
            context_parts.append(f"{header}\n{c.text}")

    elif conditions:
        meta_filter = build_meta_filter(conditions)
        hits = index.vector_search(question, k=80, meta_filter=meta_filter, expand_to_parent=True)  # ← vector_search
        for h in hits:
            context_parts.append(f"{meta_header_from_metadata(h.doc_id, doc_to_meta.get(h.doc_id, {}))}\n{h.text}")

    else:
        hits = index.vector_search(question, k=10, expand_to_parent=True)  # ← vector_search
        for h in hits:
            context_parts.append(f"{meta_header_from_metadata(h.doc_id, doc_to_meta.get(h.doc_id, {}))}\n{h.text}")

    context = "\n\n---\n\n".join(context_parts)
    final_prompt = SYSTEM_PROMPT_NEW_V2.format(context=context, question=question)

    for attempt in range(max_retries):
        response = client.chat.completions.create(
            model=model_name,
            messages=[{"role": "user", "content": final_prompt}],
            max_completion_tokens=8000,
            reasoning_effort="low"
        )
        answer = response.choices[0].message.content
        if answer:
            return answer
    return "(답변 생성 실패)"

In [69]:
# core40 - Chroma 순수 벡터 버전 답변 생성

chroma_vector_answers_40 = []
for item in core40:
    cid = item['case_id']
    task_type = item['task_type']
    question = item['question']

    if task_type == 'follow_up':
        history = item.get('history', [])
        user_turns = [h['content'] for h in history if h.get('role') == 'user']
        prev_q = user_turns[-1] if user_turns else ""
        combined_q = f"{prev_q} {question}"
        answer = ask_rfp_final_chroma_vector_only(combined_q)
    else:
        answer = ask_rfp_final_chroma_vector_only(question)

    chroma_vector_answers_40.append({'case_id': cid, 'task_type': task_type, 'answer': answer})
    print(f"[Chroma-Vector][{cid}][{task_type}] {question}")
    print(answer)
    print()

[Chroma-Vector][dev-single-001][single_doc] BIFF&ACFM 온라인서비스 재개발 사업의 사업예산은 얼마이며 부가가치세가 포함돼 있나요?
사업예산: 243,000,000원 — 부가가치세(VAT) 포함되어 있습니다.

[근거: (사)부산국제영화제_2024년 BIFF & ACFM 온라인서비스 재개발 및 행사지원시.hwp]

[Chroma-Vector][dev-single-002][single_doc] e나라도움 웹 접근성 컨설팅의 수행기간과 예산을 알려줘.
수행기간: 계약일로부터 120일(약 4개월)  
예산: 70,000,000원 (VAT 포함)

[근거: 한국재정정보원_e나라도움 업무시스템 웹 접근성 컨설팅.hwp]

[Chroma-Vector][dev-single-003][single_doc] 국립인천해양박물관 해양자료관리시스템 사업은 몇 차로 나뉘고 기술평가와 가격평가 비중은 어떻게 되나요?
- 사업은 2차로 구분됨: 1차(계약일로부터 4개월) + 2차(1차 완료일로부터 2개월).  
- 평가 비중: 기술평가 90%, 가격평가 10%.

[근거: 국립인천해양박물관_국립인천해양박물관 해양자료관리시스템 구축 용.hwp]

[Chroma-Vector][dev-single-004][single_doc] RFID 기반 국가자격 시험 결과물 관리시스템의 사업기간과 시범 도입 규모는?
사업기간: 계약일로부터 90일(계약일로부터 3개월, 기재상 2024년 11월 1일까지).  
시범 도입 규모: 제공된 문서 내에서 확인되지 않습니다.

[근거: 한국산업인력공단_RFID기반 국가자격 시험 결과물 스마트 관리시스템 도.hwp]

[Chroma-Vector][dev-single-005][single_doc] 서민금융 채팅 상담시스템 입찰에서 입찰서와 제안서는 어떻게 제출하고 파일 형식·용량 제한은 무엇인가요?
- 입찰서와 제안서는 전자적으로 제출해야 하며(나라장터를 통해), 입찰서와 제안서를 모두 제출해야 유효합니다. 입찰금액은 부가가치세를 

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

[Chroma-Vector][dev-unknown-001][unknown] 오늘 새로 올라온 나라장터 공고 세 개만 알려줘.
이 기능은 제가 수행할 수 없습니다. 실시간(오늘)로 신규 게시된 나라장터 공고 정보는 제공된 문서에서 확인할 수 없습니다.

[Chroma-Vector][dev-unknown-002][unknown] BIFF&ACFM 온라인서비스 재개발 사업의 최종 낙찰업체와 낙찰금액은 얼마야?
확인되지 않습니다.

[근거: (사)부산국제영화제_2024년 BIFF & ACFM 온라인서비스 재개발 및 행사지원시.hwp]



Batches:   0%|          | 0/1 [00:00<?, ?it/s]

[Chroma-Vector][dev-unknown-003][unknown] 공고 담당자에게 지금 전화해서 제안설명회 참석한다고 전해줘.
이 기능은 제가 수행할 수 없습니다.

[Chroma-Vector][dev-unknown-004][unknown] 우리 회사가 BIFF&ACFM 사업의 입찰참가자격을 모두 충족하는지 판정해줘.
문서만으로 귀사가 자격을 충족하는지 여부를 판정할 수 없습니다.

대신 평가(판정)를 위해 귀사에서 확인해야 할 필수 조건들을 문서 근거로 정리합니다. 각 항목을 내부 증빙(사업자등록, 본사 주소, 나라장터 등록증, 직접생산확인증명서 등)과 대조해 주세요.

필수 확인항목(문서 근거)
1. 부정당업자 해당 여부 비해당 확인
   - 지방자치단체 계약법 시행령 제92조에 해당되지 않아야 함.
2. 주된 영업소 소재지
   - 입찰공고일 전일부터 계약체결일까지 주된 영업소의 소재지가 부산광역시여야 함.
3. 나라장터(G2B) 등록
   - 입찰서 제출마감일 전일까지 나라장터에 소프트웨어사업자(업종코드 1468, 컴퓨터관련서비스사업)로 입찰참가자격 등록되어 있어야 함.
4. 중소·중견·대기업 제한
   - 소프트웨어산업 진흥법 및 관련 지침에 따라 대기업·중견기업 및 상호출자제한기업집단 소속회사는 참여 불가. 귀사가 해당되지 않아야 함.
5. 직접생산확인증명서
   - 정보시스템개발서비스(세부품명번호 8111159901) 관련 ‘직접생산확인증명서’를 입찰마감 전일까지 발급받아 유효기간 내여야 함.
6. 공동수급·하도급 금지
   - 공동수급(공동이행방식) 및 하도급은 허용되지 않음. 단독으로 수행 가능해야 함.
7. 산출내역·비용구성 준수
   - 제안서의 산출내역을 용역비(직접인건비, 기술료, 재경비, 공급비용, 부가세 등) 및 세부내용으로 구분하여 제시해야 함.
8. S/W 개발 관련 장비·작업장
   - S/W 개발에 필요한 작업장 및 장비는 제안사가 제공해야 하며 임대비를 별도 산정할 수 없음.

위 항목들 중 하나라도 충족 여부가 문

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

[Chroma-Vector][dev-unknown-007][unknown] 평가위원들이 실제로 더 중요하게 보는 비공개 평가 포인트를 알려줘.
답변할 수 없습니다.

문서 범위에서 확인되지 않는 '평가위원들의 실제(비공개) 우선순위나 판단 기준'은 제 문서들만으로는 확정할 수 없으며 추측할 수 없습니다. 정확히 알기 위해 필요한 자료(간단히):

- 해당 평가위원회의 내부 평가세부점수·평가위원 코멘트(비공개 원문)  
- 이전 평가회의 녹취록·회의록·평가보고서(익명화된 사례 포함)  
- 평가위원 또는 발주기관 담당자와의 직접 인터뷰 또는 서면 해명  
- 평가위원별 배점 변경 이력 또는 내부 운영지침 문서  
- 평가에 활용된 실제 채점표·가중치 적용 내역

위 자료가 제공되면 문서에 근거해 분석·정리해 드릴 수 있습니다.

[근거: 수협중앙회_강릉어선안전조업국 상황관제시스템 구축.hwp, 서울특별시_2024년 지도정보 플랫폼 및 전문활용 연계 시스템 고도화 용.pdf]

[Chroma-Vector][dev-unknown-008][unknown] BIFF&ACFM 사업과 서민금융 채팅상담 사업 중 우리 회사 수주확률이 더 높은 곳을 퍼센트로 계산해줘.
확정적 수주확률(퍼센트)은 제공된 문서만으로 산정할 수 없습니다. 이런 판정은 문서에 없는 경쟁사 현황·귀사 역량·제안가격 등 핵심정보가 필요하므로 답변할 수 없습니다.

대신 수주 가능성 판단을 위해 필요한 주요 조건(간단 목록):
- 귀사의 관련 과제 수행 실적(유사사업, 레퍼런스) 및 평가점수 영향 항목 충족 여부  
- 제안가격(입찰서·협상 가격) 및 비용구성 비교 자료  
- 기술적 적합성(요구기능 충족 여부, 기존 시스템 연계 가능성) — 예: 서민금융 과제는 분리발주 SW 통합 역량 요구  
- 인력·자원 가용성(파견 필요 인력 포함) 및 일정 준수 능력  
- 보안·인증 요건 충족 여부(특히 서민금융 관련 보안요건)  
- 입찰 방식·제한(지역 제한, 긴급공고 등)과 귀사 자격 적합성  
- 경쟁

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

[Chroma-Vector][dev-unknown-009][unknown] 입찰 참여 시작일이 문서에 없으면 공개일 다음 날이라고 보고 날짜를 확정해줘.
확정할 수 없습니다. 제공된 문서들에서 “입찰 참여 시작일을 공개일의 다음 날로 간주하라”는 근거가 없고, 구조화된 필드 관련 유의사항에 따르면 공개일을 입찰참가 시작일로 대체해서는 안 된다고 안내되어 있어 임의로 확정할 수 없습니다. 원문 확인이 필요합니다.

[근거: 수협중앙회_수협중앙회 수산물사이버직매장 시스템 재구축 ISMP 수립 입.hwp, 한국농어촌공사_아세안+3 식량안보정보시스템(AFSIS) 3단계 협력(캄보디아.hwp, 경희대학교_[입찰공고] 산학협력단 정보시스템 운영 용역업체 선정.hwp, 사단법인아시아물위원회사무국_우즈벡-키르기즈스탄 기후변화대응 스.hwp, 한국수출입은행_(긴급) 모잠비크 마푸토 지능형교통시스템(ITS) 구축사업.hwp, 인천광역시 동구_수도국산달동네박물관 전시해설 시스템 구축(협상에 .hwp, 한국원자력연구원_한국원자력연구원 선량평가시스템 고도화.hwp, 중앙선거관리위원회_2025년도 행정정보시스템 위탁운영사업.hwp, 한국교육과정평가원_국가교육과정정보센터(NCIC) 시스템 운영 및 개선.hwp, 부산관광공사_경영정보시스템 기능개선.hwp]

[Chroma-Vector][dev-unknown-010][unknown] BIFF&ACFM 온라인서비스 재개발 사업은 현재 몇 퍼센트 진행됐고 계약 지연 가능성은 얼마나 돼?
답변할 수 없습니다.

제공된 제안요청서(공문)에는 현재까지의 수행진행률(%)이나 실제 계약 이행 현황·진행상태, 일정 지연 발생 여부에 대한 실시간·사후 보고 자료가 포함되어 있지 않습니다. 또한 계약 지연 가능성(확률)은 문서만으로는 판단할 수 없습니다.

필요한 정보(참고용)
- 현재까지 완료된 산출물·단계별 완료보고서(마일스톤별 진행상태)
- 착수보고서·주간/월간 진행 리포트 또는 공사·개발 진척률 데이터
- 변경(추가)요구사항 및 일정 변경 내역(변경

In [70]:
# rag-56 - Chroma 순수 벡터 버전 답변 생성

chroma_vector_answers_56 = []
for item in rag56:
    cid = item['case_id']
    task_type = item['task_type']
    question = item['question']

    answer = ask_rfp_final_chroma_vector_only(question)
    chroma_vector_answers_56.append({'case_id': cid, 'task_type': task_type, 'question': question, 'answer': answer})
    print(f"[Chroma-Vector][{cid}][{task_type}] {question}")
    print(answer)
    print()

[Chroma-Vector][supplemental-qa-c01][single_doc] 평택시가 정류장 이용객에게 실시간 운행정보를 제공하려고 추진한 2024년 사업에는 예산이 얼마나 배정됐나요?
￦999,494,600원 (부가세 포함).

[근거: 경기도 평택시_2024년도 평택시 버스정보시스템(BIS) 구축사업.hwp]

[Chroma-Vector][supplemental-qa-c02][single_doc] 평택시의 실시간 버스 도착정보 제공 체계를 새로 만드는 2024년 사업은 착수 후 언제까지 완료해야 하나요?
착수일로부터 2024년 10월 31일까지 완료해야 합니다.

[근거: 경기도 평택시_2024년도 평택시 버스정보시스템(BIS) 구축사업.hwp]

[Chroma-Vector][supplemental-qa-c03][single_doc] GKL이 임직원 협업과 전자결재 환경을 새로 구축하려는 2024년 정보화 사업에 책정한 금액은 얼마인가요?
1,515,000,000원 (부가세 포함).  
[근거: 그랜드코리아레저(주)_2024년도 GKL  그룹웨어 시스템 구축 용역.hwp]

[Chroma-Vector][supplemental-qa-c04][single_doc] GKL이 사내 협업 업무 환경을 새로 만드는 사업은 어떤 경쟁 방식과 낙찰 절차로 계약하나요?
경쟁 방식: 제한경쟁입찰  
낙찰 절차(사업자 선정 방식): 협상에 의한 계약 (협상에 따라 사업자 선정)

[근거: 그랜드코리아레저(주)_2024년도 GKL  그룹웨어 시스템 구축 용역.hwp]

[Chroma-Vector][supplemental-qa-c05][single_doc] 한국농어촌공사가 네팔에서 물 관리 정보를 디지털화하기 위해 추진하는 시범 시스템은 착수 후 얼마 동안 수행하나요?
용역착수일로부터 6개월간 수행합니다.

[근거: 한국농어촌공사_네팔 수자원관리 정보화사업-Pilot 시스템 구축용역.hwp]

[Chroma-Vector][supplemental-qa-c06][singl

In [71]:
chroma_vector_results_40 = score_core40_v2(chroma_vector_answers_40, core40)
summarize_score(chroma_vector_results_40, "Chroma (순수 벡터)")

Chroma (순수 벡터)
전체 평균: 88.96/100 (40개)
single_doc: 평균 89.17/100 (10개)
multi_doc_compare: 평균 75.00/100 (10개)
follow_up: 평균 91.67/100 (10개)
unknown: 평균 100.00/100 (10개)



In [72]:
chroma_vector_results_56 = score_answers(chroma_vector_answers_56, rag56)
summarize(chroma_vector_results_56, "Chroma (순수 벡터)")

Chroma (순수 벡터)
평균 fact_score: 62.56 (54건)
기권 판단 일치율: 94.64% (56건)
인용 커버리지: 79.69% (51/64)



In [73]:
# FAISS용 BM25 인덱스 준비

from rank_bm25 import BM25Okapi
from src.retrieval.indexing import _tokenize_ko

faiss_tokenized_corpus = [_tokenize_ko(c.text) for c in child_chunks]
faiss_bm25 = BM25Okapi(faiss_tokenized_corpus)
print("FAISS용 BM25 인덱스 준비 완료")

FAISS용 BM25 인덱스 준비 완료


In [74]:
# FAISS + BM25 하이브리드 검색 함수

def _minmax(scores):
    if scores.size == 0:
        return scores
    lo, hi = scores.min(), scores.max()
    if hi - lo < 1e-9:
        return np.ones_like(scores)
    return (scores - lo) / (hi - lo)

def faiss_hybrid_search(query, k=5, candidate_k=20, vector_weight=0.5, bm25_weight=0.5, allowed_docs=None):
    # dense 후보
    query_vec = index.embedding_backend.encode([query])
    search_k = min(len(child_chunks), 2000)
    distances, indices = index_faiss.search(np.array(query_vec).astype('float32'), search_k)

    v_candidates = []
    for idx, dist in zip(indices[0], distances[0]):
        c = faiss_chunk_lookup[idx]
        if allowed_docs is not None and c.doc_id not in allowed_docs:
            continue
        v_candidates.append((c, 1 - dist))
        if len(v_candidates) >= candidate_k:
            break

    # BM25 후보
    tokens = _tokenize_ko(query)
    bm25_scores = faiss_bm25.get_scores(tokens)
    order = np.argsort(bm25_scores)[::-1]
    b_candidates = []
    for idx in order:
        if bm25_scores[idx] <= 0:
            break
        c = child_chunks[idx]
        if allowed_docs is not None and c.doc_id not in allowed_docs:
            continue
        b_candidates.append((c, float(bm25_scores[idx])))
        if len(b_candidates) >= candidate_k:
            break

    v_scores = _minmax(np.array([s for _, s in v_candidates])) if v_candidates else np.array([])
    b_scores = _minmax(np.array([s for _, s in b_candidates])) if b_candidates else np.array([])

    combined = {}
    combined_score = {}
    for (c, _), s in zip(v_candidates, v_scores):
        combined[c.chunk_id] = c
        combined_score[c.chunk_id] = combined_score.get(c.chunk_id, 0.0) + vector_weight * float(s)
    for (c, _), s in zip(b_candidates, b_scores):
        combined[c.chunk_id] = c
        combined_score[c.chunk_id] = combined_score.get(c.chunk_id, 0.0) + bm25_weight * float(s)

    ranked_ids = sorted(combined_score, key=combined_score.get, reverse=True)[:k]
    results = [{'doc_id': combined[cid].doc_id, 'text': combined[cid].text} for cid in ranked_ids]
    return results

In [75]:
# 버전 D: FAISS + BM25 Hybrid ask_rfp_final

def ask_rfp_final_faiss_hybrid(question, model_name="gpt-5-mini", max_retries=2):
    doc_hints = extract_doc_hints_multi(question, all_filenames_with_biz)
    doc_hints = doc_hints[:3]
    keywords = find_relevant_keywords(question)
    conditions = extract_filter_conditions(question)

    doc_to_meta = {}
    for c in child_chunks:
        if c.doc_id not in doc_to_meta:
            doc_to_meta[c.doc_id] = c.metadata

    context_parts = []

    def get_doc_chunks(doc_id):
        return [c for c in child_chunks if c.doc_id == doc_id]

    if is_aggregation_question(question) and len(doc_hints) >= 1:
        stopwords_q = {'사업의', '사업에서', '어떻게', '되나요?', '되나요', '몇', '어떤', '얼마', '비교'}
        qkeywords = [w for w in re.split(r'[ ,]', question) if len(w) >= 2 and w not in stopwords_q]
        best_doc, best_score = doc_hints[0], -1
        for fname in doc_hints:
            biz = doc_to_meta.get(fname, {}).get('발주_기관', '')
            score = sum(1 for kw in qkeywords if kw in fname or kw in str(biz))
            if score > best_score:
                best_score, best_doc = score, fname
        doc_hint = best_doc
        header = meta_header_from_metadata(doc_hint, doc_to_meta.get(doc_hint, {}))
        for c in get_doc_chunks(doc_hint):
            context_parts.append(f"{header}\n{c.text}")

    elif len(doc_hints) == 1 and keywords:
        doc_hint = doc_hints[0]
        doc_c = get_doc_chunks(doc_hint)
        keyword_chunks = [c for c in doc_c if any(kw in c.text for kw in keywords)]
        header = meta_header_from_metadata(doc_hint, doc_to_meta.get(doc_hint, {}))
        if keyword_chunks:
            for c in keyword_chunks[:25]:
                context_parts.append(f"{header}\n{c.text}")
        else:
            results = faiss_hybrid_search(question, k=10)
            for r in results:
                context_parts.append(f"{meta_header_from_metadata(r['doc_id'], doc_to_meta.get(r['doc_id'], {}))}\n{r['text']}")

    elif len(doc_hints) >= 2:
        for doc_hint in doc_hints:
            doc_c = get_doc_chunks(doc_hint)
            if keywords:
                matched = [c for c in doc_c if any(kw in c.text for kw in keywords)]
                selected = matched[:8] if matched else doc_c[:8]
            else:
                selected = doc_c[:8]
            header = meta_header_from_metadata(doc_hint, doc_to_meta.get(doc_hint, {}))
            for c in selected:
                context_parts.append(f"{header}\n{c.text}")

    elif doc_hints:
        doc_hint = doc_hints[0]
        doc_c = get_doc_chunks(doc_hint)
        header = meta_header_from_metadata(doc_hint, doc_to_meta.get(doc_hint, {}))
        for c in doc_c[:15]:
            context_parts.append(f"{header}\n{c.text}")

    elif conditions:
        allowed_docs = None
        if '금액_최소' in conditions or conditions.get('지자체') or conditions.get('공사'):
            allowed_docs = set()
            for c in child_chunks:
                m = c.metadata
                ok = True
                if '금액_최소' in conditions:
                    amt = m.get('사업_금액')
                    if amt is None or amt < conditions['금액_최소']:
                        ok = False
                if conditions.get('지자체') and not is_local_gov(m.get('발주_기관')):
                    ok = False
                if conditions.get('공사') and '공사' not in str(m.get('발주_기관', '')):
                    ok = False
                if ok:
                    allowed_docs.add(c.doc_id)
        results = faiss_hybrid_search(question, k=80, allowed_docs=allowed_docs)
        for r in results:
            context_parts.append(f"{meta_header_from_metadata(r['doc_id'], doc_to_meta.get(r['doc_id'], {}))}\n{r['text']}")

    else:
        results = faiss_hybrid_search(question, k=10)
        for r in results:
            context_parts.append(f"{meta_header_from_metadata(r['doc_id'], doc_to_meta.get(r['doc_id'], {}))}\n{r['text']}")

    context = "\n\n---\n\n".join(context_parts)
    final_prompt = SYSTEM_PROMPT_NEW_V2.format(context=context, question=question)

    for attempt in range(max_retries):
        response = client.chat.completions.create(
            model=model_name,
            messages=[{"role": "user", "content": final_prompt}],
            max_completion_tokens=8000,
            reasoning_effort="low"
        )
        answer = response.choices[0].message.content
        if answer:
            return answer
    return "(답변 생성 실패)"

In [76]:
# core40 - FAISS+BM25 Hybrid 답변 생성

faiss_hybrid_answers_40 = []
for item in core40:
    cid = item['case_id']
    task_type = item['task_type']
    question = item['question']

    if task_type == 'follow_up':
        history = item.get('history', [])
        user_turns = [h['content'] for h in history if h.get('role') == 'user']
        prev_q = user_turns[-1] if user_turns else ""
        combined_q = f"{prev_q} {question}"
        answer = ask_rfp_final_faiss_hybrid(combined_q)
    else:
        answer = ask_rfp_final_faiss_hybrid(question)

    faiss_hybrid_answers_40.append({'case_id': cid, 'task_type': task_type, 'answer': answer})
    print(f"[FAISS-Hybrid][{cid}][{task_type}] {question}")
    print(answer)
    print()

[FAISS-Hybrid][dev-single-001][single_doc] BIFF&ACFM 온라인서비스 재개발 사업의 사업예산은 얼마이며 부가가치세가 포함돼 있나요?
사업예산은 금 243,000,000원이며, 부가가치세(VAT) 포함으로 표기되어 있습니다.

[근거: (사)부산국제영화제_2024년 BIFF & ACFM 온라인서비스 재개발 및 행사지원시.hwp]

[FAISS-Hybrid][dev-single-002][single_doc] e나라도움 웹 접근성 컨설팅의 수행기간과 예산을 알려줘.
수행기간: 계약일로부터 120일(약 4개월)
예산: 70,000,000원 (VAT 포함)

[근거: 한국재정정보원_e나라도움 업무시스템 웹 접근성 컨설팅.hwp]

[FAISS-Hybrid][dev-single-003][single_doc] 국립인천해양박물관 해양자료관리시스템 사업은 몇 차로 나뉘고 기술평가와 가격평가 비중은 어떻게 되나요?
- 사업은 2차(1차: 계약일로부터 4개월, 2차: 1차 완료일로부터 2개월)로 나뉩니다.
- 평가 비중은 기술평가 90%, 가격평가 10%입니다.

[근거: 국립인천해양박물관_국립인천해양박물관 해양자료관리시스템 구축 용.hwp]

[FAISS-Hybrid][dev-single-004][single_doc] RFID 기반 국가자격 시험 결과물 관리시스템의 사업기간과 시범 도입 규모는?
사업기간: 계약일로부터 90일.  
시범 도입 규모: 제공된 문서 범위에서는 확인되지 않습니다.

[근거: 한국산업인력공단_RFID기반 국가자격 시험 결과물 스마트 관리시스템 도.hwp]

[FAISS-Hybrid][dev-single-005][single_doc] 서민금융 채팅 상담시스템 입찰에서 입찰서와 제안서는 어떻게 제출하고 파일 형식·용량 제한은 무엇인가요?
- 제출 방식(핵심)
  - 입찰서: 나라장터를 통해 전자적으로만 제출해야 함. 제안서와 입찰서를 모두 제출해야 입찰이 유효함. (입찰서는 전자제출 우선)  
  - 기타 제출서류: 나

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

[FAISS-Hybrid][dev-unknown-001][unknown] 오늘 새로 올라온 나라장터 공고 세 개만 알려줘.
이 기능은 제가 수행할 수 없습니다. 제공된 문서만으로는 “오늘”(실시간 기준) 새로 올라온 나라장터 공고 여부를 확인할 수 없습니다.

[FAISS-Hybrid][dev-unknown-002][unknown] BIFF&ACFM 온라인서비스 재개발 사업의 최종 낙찰업체와 낙찰금액은 얼마야?
확인되지 않습니다.

[근거: (사)부산국제영화제_2024년 BIFF & ACFM 온라인서비스 재개발 및 행사지원시.hwp]



Batches:   0%|          | 0/1 [00:00<?, ?it/s]

[FAISS-Hybrid][dev-unknown-003][unknown] 공고 담당자에게 지금 전화해서 제안설명회 참석한다고 전해줘.
이 기능은 제가 수행할 수 없습니다.

[FAISS-Hybrid][dev-unknown-004][unknown] 우리 회사가 BIFF&ACFM 사업의 입찰참가자격을 모두 충족하는지 판정해줘.
이 문서만으로 귀사가 자격을 충족하는지 판정할 수 없습니다.

대신 문서에서 요구하는 입찰참가자격 조건을 아래에 모두 나열하니, 귀사 현황과 대조해 충족 여부를 직접 확인해 주세요.

필수 참가자격(문서 근거)
1. 부정당업자 제한 규정 해당 여부
   - 지방자치단체를 당사자로 하는 계약에 관한 법령 시행령 제92조에 해당되지 않아야 함.

2. 주된 영업소 소재지
   - 입찰공고일 전일부터 계약체결일까지 주된 영업소의 소재지가 부산광역시인 기업(동법 시행령 제20조 제1항 6호 및 시행규칙 제24조 제2항).

3. 나라장터(G2B) 등록
   - 입찰서 제출마감일 전일까지 나라장터에 ‘소프트웨어사업자[컴퓨터관련서비스사업(업종코드 1468)]’로 입찰참가자격 등록 완료.

4. 기업형태(규모) 제한
   - 소프트웨어산업 진흥법(제48조) 및 중소소프트웨어사업자 참여지원 지침(제2조·제3조)에 따라 대기업·중견기업 및 상호출자제한기업집단 소속회사는 참여 불가(즉, 중소 소프트웨어사업자여야 함).

5. 직접생산확인증명서
   - ‘정보시스템개발서비스(세부품명번호 8111159901)’의 직접생산확인증명서를 보유(입찰마감 전일까지 발급된 유효한 것).

6. 공동수급 및 하도급 금지
   - 공동수급(공동이행방식) 불허, 하도급 허용되지 않음(단독 참여만 허용).

7. 제반 준수·윤리 약정
   - 제출서류의 진실성, 청렴계약 이행서약서 등 계약·윤리 관련 약정 사항 준수 가능성(담합·뇌물 등 위반 시 제재 규정 존재).

확인 필요 항목(문서 내 명확 근거는 있으나 귀사 내부정보가 필요)
- 귀사의 주된 영업소 소재지가 부산인지(기간

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

[FAISS-Hybrid][dev-unknown-007][unknown] 평가위원들이 실제로 더 중요하게 보는 비공개 평가 포인트를 알려줘.
확인되지 않습니다.

제공된 문서들은 평가항목·세부기준·결과를 비공개로 하거나 공개하지 않으며(예: 평가기준·배점은 위원회 결정에 따름, 세부기준 및 결과 비공개 등), 문서 범위 내에서는 평가위원들이 '실제로 더 중요하게 보는 비공개 평가 포인트'를 특정할 근거가 없습니다. 원문 전체(또는 평가위원회에서 공개한 세부자료)가 있어야만 확인 가능합니다.

[근거: 수협중앙회_강릉어선안전조업국 상황관제시스템 구축.hwp, 조선대학교_(재공고)2024 조선대학교 SW중심대학 사업관리시스템(WeHub) 구.hwp, 부산관광공사_경영정보시스템 기능개선.hwp]

[FAISS-Hybrid][dev-unknown-008][unknown] BIFF&ACFM 사업과 서민금융 채팅상담 사업 중 우리 회사 수주확률이 더 높은 곳을 퍼센트로 계산해줘.
문서만으로 수주확률(퍼센트)을 계산하는 것은 불가능합니다. 이런 판정은 제안사(귀사)의 구체적 능력·가격·입찰전략·경쟁구도 등 주관적·외부요인이 필요하므로 문서 범위만으로 확정적인 확률을 제시할 수 없습니다.

판단에 필요한 주요 항목(간단 목록)
- 귀사 기술역량·유사 수행실적(레퍼런스) 및 인력 구성
- 제안서(기술·가격) 제출 내용 및 경쟁력(가격, 기술·기능 충족여부)
- 입찰참가자격(지역·자격 제한 등) 및 귀사의 적합성
- 입찰 방식·평가기준(배점 구조, 기술·가격 비중) — 문서에서 확인된 평가 기준 세부사항
- 경쟁사 수(및 알려진 주요 경쟁사 역량)
- 계약조건·제약사항(보증금·하자책임·분리발주 등)과 귀사의 수용여부
- 일정(제출기한·발표일 등)과 귀사의 준비상태

위 항목들을 귀사 내부 자료 및 경쟁상황과 대조한 후에야 수주 가능성(확률)을 산정할 수 있습니다. 필요하면 위 항목별로 어떤 자료를 준비해야 하는지 정리해 드리겠습니다.

[근거: 서민금융진흥원_서민금융진흥원 서민

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

[FAISS-Hybrid][dev-unknown-009][unknown] 입찰 참여 시작일이 문서에 없으면 공개일 다음 날이라고 보고 날짜를 확정해줘.
확인되지 않습니다. 제공된 문서들에는 “입찰 참여 시작일이 문서에 없으면 공개일 다음 날로 본다”는 규정이나 지침이 명시되어 있지 않습니다. 각 문서들은 입찰참가자격의 판단기준일을 “입찰공고문에 따름”이라고만 규정하고 있어(따라서 실제 시작일은 해당 입찰공고문을 확인해야 함), 임의로 공개일 다음 날로 확정할 근거가 없습니다. 원문 공고문을 확인해 주세요.

[근거: 수협중앙회_수협중앙회 수산물사이버직매장 시스템 재구축 ISMP 수립 입.hwp, 경희대학교_[입찰공고] 산학협력단 정보시스템 운영 용역업체 선정.hwp]

[FAISS-Hybrid][dev-unknown-010][unknown] BIFF&ACFM 온라인서비스 재개발 사업은 현재 몇 퍼센트 진행됐고 계약 지연 가능성은 얼마나 돼?
실시간 진행률과 향후 계약 지연 확률에 대해서는 제공된 문서에 근거한 확인이 불가능합니다.

- 현재 진행률: 실시간 상태(몇 % 완료 등)는 제안요청서 문서에 명시되어 있지 않습니다. 이 기능은 제가 수행할 수 없습니다.
- 계약 지연 가능성(확률): 문서만으로는 향후 일정 이행 리스크나 지연 확률을 산정할 수 없습니다. 이런 판단은 실제 진행상황, 납품·검수 이력, 인력 파견 현황, 계약서 상의 일정·위약금 조항, 발주처·수급사 간 커뮤니케이션 등 구체적 증거가 필요합니다.

필요하면, 지연 위험 판단에 필요한 최소 항목들을 간단히 정리해 드릴 수 있습니다(예: 현재 산출물 제출 현황, 파견 인력 상주 여부 등). 원하시면 알려주세요.

[근거: (사)부산국제영화제_2024년 BIFF & ACFM 온라인서비스 재개발 및 행사지원시.hwp]



In [77]:
# rag-56 - FAISS+BM25 Hybrid 답변 생성

faiss_hybrid_answers_56 = []
for item in rag56:
    cid = item['case_id']
    task_type = item['task_type']
    question = item['question']

    answer = ask_rfp_final_faiss_hybrid(question)
    faiss_hybrid_answers_56.append({'case_id': cid, 'task_type': task_type, 'question': question, 'answer': answer})
    print(f"[FAISS-Hybrid][{cid}][{task_type}] {question}")
    print(answer)
    print()

[FAISS-Hybrid][supplemental-qa-c01][single_doc] 평택시가 정류장 이용객에게 실시간 운행정보를 제공하려고 추진한 2024년 사업에는 예산이 얼마나 배정됐나요?
예산: ￦999,494,600원(부가세 포함)

[근거: 경기도 평택시_2024년도 평택시 버스정보시스템(BIS) 구축사업.hwp]

[FAISS-Hybrid][supplemental-qa-c02][single_doc] 평택시의 실시간 버스 도착정보 제공 체계를 새로 만드는 2024년 사업은 착수 후 언제까지 완료해야 하나요?
착수일로부터 2024년 10월 31일까지 완료해야 합니다.

[근거: 경기도 평택시_2024년도 평택시 버스정보시스템(BIS) 구축사업.hwp]

[FAISS-Hybrid][supplemental-qa-c03][single_doc] GKL이 임직원 협업과 전자결재 환경을 새로 구축하려는 2024년 정보화 사업에 책정한 금액은 얼마인가요?
사업예산: 1,515,000천원 (부가세 포함).  
[근거: 그랜드코리아레저(주)_2024년도 GKL  그룹웨어 시스템 구축 용역.hwp]

[FAISS-Hybrid][supplemental-qa-c04][single_doc] GKL이 사내 협업 업무 환경을 새로 만드는 사업은 어떤 경쟁 방식과 낙찰 절차로 계약하나요?
제한경쟁입찰로 공고하고, 사업자 선정은 협상에 의한 계약(협상계약) 방식으로 체결합니다.  
[근거: 그랜드코리아레저(주)_2024년도 GKL  그룹웨어 시스템 구축 용역.hwp]

[FAISS-Hybrid][supplemental-qa-c05][single_doc] 한국농어촌공사가 네팔에서 물 관리 정보를 디지털화하기 위해 추진하는 시범 시스템은 착수 후 얼마 동안 수행하나요?
용역착수일로부터 6개월입니다.

[근거: 한국농어촌공사_네팔 수자원관리 정보화사업-Pilot 시스템 구축용역.hwp]

[FAISS-Hybrid][supplemental-qa-c06][single_doc] 한국농어촌공사

In [78]:
# 채점: core40, rag-56

faiss_hybrid_results_40 = score_core40_v2(faiss_hybrid_answers_40, core40)
summarize_score(faiss_hybrid_results_40, "FAISS+BM25 Hybrid")

faiss_hybrid_results_56 = score_answers(faiss_hybrid_answers_56, rag56)
summarize(faiss_hybrid_results_56, "FAISS+BM25 Hybrid")

FAISS+BM25 Hybrid
전체 평균: 86.67/100 (40개)
single_doc: 평균 89.17/100 (10개)
multi_doc_compare: 평균 70.83/100 (10개)
follow_up: 평균 86.67/100 (10개)
unknown: 평균 100.00/100 (10개)

FAISS+BM25 Hybrid
평균 fact_score: 62.56 (54건)
기권 판단 일치율: 94.64% (56건)
인용 커버리지: 78.12% (50/64)



In [79]:
from src.retrieval.indexing import HybridIndex, SearchHit, _tokenize_ko
from rank_bm25 import BM25Okapi
import numpy as np

class FaissHybridIndex(HybridIndex):
    """HybridIndex와 완전히 동일한 검색 로직 - Chroma 생성만 건너뛰고 FAISS로 대체"""
    def __init__(self, chunks, embedding_backend=None):
        # 부모 __init__을 호출하지 않고, 필요한 속성만 직접 설정
        # (Chroma 컬렉션 생성 부분만 건너뜀 - bm25_search/hybrid_search/_expand_hits_to_parent는
        #  전부 부모 클래스 메서드를 그대로 상속해서 씀, 코드 한 글자도 안 바뀜)
        self.chunks = chunks
        self.by_id = {c.chunk_id: c for c in chunks}
        self._searchable_chunks = [c for c in chunks if c.strategy != "parent"]
        self.embedding_backend = embedding_backend or index.embedding_backend

        # BM25 인덱스 (부모 __init__과 완전히 동일한 방식으로 구축)
        self._tokenized_corpus = [_tokenize_ko(c.text) for c in self._searchable_chunks]
        self._bm25 = BM25Okapi(self._tokenized_corpus) if self._searchable_chunks else None

    def vector_search(self, query, k=5, meta_filter=None, expand_to_parent=False):
        # 여기만 Chroma 대신 FAISS 사용 - 나머지 로직(필터링, 결과 개수 제한 등)은
        # 부모 클래스의 vector_search와 동일한 흐름
        query_vec = self.embedding_backend.encode([query])
        search_k = min(len(self._searchable_chunks), 2000)
        distances, indices = index_faiss.search(np.array(query_vec).astype('float32'), search_k)

        hits = []
        for idx, dist in zip(indices[0], distances[0]):
            chunk = faiss_chunk_lookup[idx]
            if meta_filter and not meta_filter(chunk.metadata):
                continue
            hits.append(SearchHit(chunk.chunk_id, chunk.doc_id, chunk.text, chunk.metadata, score=1 - dist, matched_by="vector"))
            if len(hits) >= k:
                break
        return self._expand_hits_to_parent(hits) if expand_to_parent else hits

In [80]:
faiss_index = FaissHybridIndex(chunks)
print(f"검색 대상(child) chunk 수: {len(faiss_index._searchable_chunks)}")

검색 대상(child) chunk 수: 14575


In [81]:
# 버전 D-v2: FAISS + BM25/hybrid_search

def ask_rfp_final_faiss_hybrid_v2(question, model_name="gpt-5-mini", max_retries=2):
    doc_hints = extract_doc_hints_multi(question, all_filenames_with_biz)
    doc_hints = doc_hints[:3]
    keywords = find_relevant_keywords(question)
    conditions = extract_filter_conditions(question)

    doc_to_meta = {}
    for c in child_chunks:
        if c.doc_id not in doc_to_meta:
            doc_to_meta[c.doc_id] = c.metadata

    context_parts = []

    def get_doc_chunks(doc_id):
        return [c for c in child_chunks if c.doc_id == doc_id]

    def build_meta_filter(conds):
        if not conds:
            return None
        def _filter(meta):
            if '금액_최소' in conds:
                amt = meta.get('사업_금액')
                if amt is None or amt < conds['금액_최소']:
                    return False
            if conds.get('지자체'):
                if not is_local_gov(meta.get('발주_기관')):
                    return False
            if conds.get('공사'):
                org = str(meta.get('발주_기관', ''))
                if '공사' not in org:
                    return False
            return True
        return _filter

    if is_aggregation_question(question) and len(doc_hints) >= 1:
        stopwords_q = {'사업의', '사업에서', '어떻게', '되나요?', '되나요', '몇', '어떤', '얼마', '비교'}
        qkeywords = [w for w in re.split(r'[ ,]', question) if len(w) >= 2 and w not in stopwords_q]
        best_doc, best_score = doc_hints[0], -1
        for fname in doc_hints:
            biz = doc_to_meta.get(fname, {}).get('발주_기관', '')
            score = sum(1 for kw in qkeywords if kw in fname or kw in str(biz))
            if score > best_score:
                best_score, best_doc = score, fname
        doc_hint = best_doc
        header = meta_header_from_metadata(doc_hint, doc_to_meta.get(doc_hint, {}))
        for c in get_doc_chunks(doc_hint):
            context_parts.append(f"{header}\n{c.text}")

    elif len(doc_hints) == 1 and keywords:
        doc_hint = doc_hints[0]
        doc_c = get_doc_chunks(doc_hint)
        keyword_chunks = [c for c in doc_c if any(kw in c.text for kw in keywords)]
        header = meta_header_from_metadata(doc_hint, doc_to_meta.get(doc_hint, {}))
        if keyword_chunks:
            for c in keyword_chunks[:25]:
                context_parts.append(f"{header}\n{c.text}")
        else:
            hits = faiss_index.hybrid_search(question, k=10, expand_to_parent=True)
            for h in hits:
                context_parts.append(f"{meta_header_from_metadata(h.doc_id, doc_to_meta.get(h.doc_id, {}))}\n{h.text}")

    elif len(doc_hints) >= 2:
        for doc_hint in doc_hints:
            doc_c = get_doc_chunks(doc_hint)
            if keywords:
                matched = [c for c in doc_c if any(kw in c.text for kw in keywords)]
                selected = matched[:8] if matched else doc_c[:8]
            else:
                selected = doc_c[:8]
            header = meta_header_from_metadata(doc_hint, doc_to_meta.get(doc_hint, {}))
            for c in selected:
                context_parts.append(f"{header}\n{c.text}")

    elif doc_hints:
        doc_hint = doc_hints[0]
        doc_c = get_doc_chunks(doc_hint)
        header = meta_header_from_metadata(doc_hint, doc_to_meta.get(doc_hint, {}))
        for c in doc_c[:15]:
            context_parts.append(f"{header}\n{c.text}")

    elif conditions:
        meta_filter = build_meta_filter(conditions)
        hits = faiss_index.hybrid_search(question, k=80, meta_filter=meta_filter, expand_to_parent=True)
        for h in hits:
            context_parts.append(f"{meta_header_from_metadata(h.doc_id, doc_to_meta.get(h.doc_id, {}))}\n{h.text}")

    else:
        hits = faiss_index.hybrid_search(question, k=10, expand_to_parent=True)
        for h in hits:
            context_parts.append(f"{meta_header_from_metadata(h.doc_id, doc_to_meta.get(h.doc_id, {}))}\n{h.text}")

    context = "\n\n---\n\n".join(context_parts)
    final_prompt = SYSTEM_PROMPT_NEW_V2.format(context=context, question=question)

    for attempt in range(max_retries):
        response = client.chat.completions.create(
            model=model_name,
            messages=[{"role": "user", "content": final_prompt}],
            max_completion_tokens=8000,
            reasoning_effort="low"
        )
        answer = response.choices[0].message.content
        if answer:
            return answer
    return "(답변 생성 실패)"

In [82]:
# core40 - FAISS + 팀원 hybrid_search 로직 답변 생성

faiss_hybrid_v2_answers_40 = []
for item in core40:
    cid = item['case_id']
    task_type = item['task_type']
    question = item['question']

    if task_type == 'follow_up':
        history = item.get('history', [])
        user_turns = [h['content'] for h in history if h.get('role') == 'user']
        prev_q = user_turns[-1] if user_turns else ""
        combined_q = f"{prev_q} {question}"
        answer = ask_rfp_final_faiss_hybrid_v2(combined_q)
    else:
        answer = ask_rfp_final_faiss_hybrid_v2(question)

    faiss_hybrid_v2_answers_40.append({'case_id': cid, 'task_type': task_type, 'answer': answer})
    print(f"[FAISS-HybridV2][{cid}][{task_type}] {question}")
    print(answer)
    print()

[FAISS-HybridV2][dev-single-001][single_doc] BIFF&ACFM 온라인서비스 재개발 사업의 사업예산은 얼마이며 부가가치세가 포함돼 있나요?
사업예산은 금 243,000,000원이며, 부가가치세(VAT) 포함으로 표기되어 있습니다.

[근거: (사)부산국제영화제_2024년 BIFF & ACFM 온라인서비스 재개발 및 행사지원시.hwp]

[FAISS-HybridV2][dev-single-002][single_doc] e나라도움 웹 접근성 컨설팅의 수행기간과 예산을 알려줘.
수행기간: 계약일로부터 120일(약 4개월)  
예산: 70,000,000원 (VAT 포함)

[근거: 한국재정정보원_e나라도움 업무시스템 웹 접근성 컨설팅.hwp]

[FAISS-HybridV2][dev-single-003][single_doc] 국립인천해양박물관 해양자료관리시스템 사업은 몇 차로 나뉘고 기술평가와 가격평가 비중은 어떻게 되나요?
- 사업은 2차사업(1차: 계약일로부터 4개월, 2차: 추가 2개월)으로 나뉩니다.
- 평가 비중은 기술평가 90%, 가격평가 10%입니다.

[근거: 국립인천해양박물관_국립인천해양박물관 해양자료관리시스템 구축 용.hwp]

[FAISS-HybridV2][dev-single-004][single_doc] RFID 기반 국가자격 시험 결과물 관리시스템의 사업기간과 시범 도입 규모는?
- 사업기간: 계약일로부터 90일(문서상 표기: 계약일로부터 3개월, 완료기한 2024.11.1까지).  
- 시범 도입 규모: 제공된 문서 범위에서는 확인되지 않습니다.

[근거: 한국산업인력공단_RFID기반 국가자격 시험 결과물 스마트 관리시스템 도.hwp]

[FAISS-HybridV2][dev-single-005][single_doc] 서민금융 채팅 상담시스템 입찰에서 입찰서와 제안서는 어떻게 제출하고 파일 형식·용량 제한은 무엇인가요?
- 제출방법: 입찰서는 나라장터를 통해 전자적으로만 제출해야 하며(입찰서와 제안서를 모두 제출해

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

[FAISS-HybridV2][dev-unknown-001][unknown] 오늘 새로 올라온 나라장터 공고 세 개만 알려줘.
이 기능은 제가 수행할 수 없습니다. 실시간(오늘) 나라장터 공고 조회는 제공된 문서에서 확인할 수 없습니다.

[FAISS-HybridV2][dev-unknown-002][unknown] BIFF&ACFM 온라인서비스 재개발 사업의 최종 낙찰업체와 낙찰금액은 얼마야?
확인되지 않습니다.

[근거: (사)부산국제영화제_2024년 BIFF & ACFM 온라인서비스 재개발 및 행사지원시.hwp]



Batches:   0%|          | 0/1 [00:00<?, ?it/s]

[FAISS-HybridV2][dev-unknown-003][unknown] 공고 담당자에게 지금 전화해서 제안설명회 참석한다고 전해줘.
이 기능은 제가 수행할 수 없습니다.

[FAISS-HybridV2][dev-unknown-004][unknown] 우리 회사가 BIFF&ACFM 사업의 입찰참가자격을 모두 충족하는지 판정해줘.
판단할 수 없습니다.

문서만으로 귀사가 실제로 자격요건을 충족하는지 여부를 판단하는 것은 제 권한과 범위를 벗어납니다. 대신 귀사가 스스로 대조·확인할 수 있도록 본 입찰의 참가자격(검토해야 할 조건)을 문서 근거로 정리합니다. 아래 항목을 모두 내부 자료(사업자등록, 주된 영업소 주소, 나라장터 등록증 등)와 대조해 확인하세요.

필수 검토 항목(증빙 필요)
1) 부정당업자 해당 여부 없음
   - 지방자치단체를 당사자로 하는 계약에 관한 법령상 부정당업자 해당 여부 확인

2) 주된 영업소 소재지
   - 입찰공고일 전일부터 계약체결일까지 주된 영업소가 부산광역시에 소재해야 함

3) 나라장터(G2B) 참가자격 등록
   - 입찰서 제출 마감일까지 나라장터에 소프트웨어사업자(컴퓨터관련서비스사업, 업종코드 1468)로 등록되어 있어야 함

4) 기업 규모 제한
   - 소프트웨어산업 진흥법 및 관련 지침에 따라 대기업·중견기업 및 상호출자제한기업집단 소속 회사는 참여 불가(중소기업 여부 확인 필요)

5) 중소기업제품 직접생산확인증명서
   - 정보시스템개발서비스(세부품명번호 8111159901) 관련 ‘직접생산확인증명서’를 입찰마감 전일까지 발급받아 유효기간 내에 있어야 함

6) 공동수급·하도급 불허
   - 공동수급(공동이행) 불가, 하도급 허용되지 않음 — 단독으로 수행 가능해야 함

7) 제출서류·증빙의 성실성
   - 제출되는 증빙서류가 허위가 아니어야 하며, 관련 청렴·윤리 서약 내용 준수 가능 여부

확인 방법(권장)
- 각 항목별 관련 증빙(법인등기, 사업자등록증, 사업장 주소 증빙, 나라장터 등록증, 직접생산확인

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

[FAISS-HybridV2][dev-unknown-007][unknown] 평가위원들이 실제로 더 중요하게 보는 비공개 평가 포인트를 알려줘.
답변할 수 없습니다.

제공된 문서들은 평가 세부내용·평가결과를 비공개로 하고 있어(평가결과 세부 비공개·이의 제기 불가 등), 문서 밖의 ‘실제 평가위원 개인의 우선순위(비공개 포인트)’를 확인할 근거가 없어 답변할 수 없습니다. (문서 근거: 평가세부내용 비공개 명시)

참고로, 만약 내부적 우선순위를 확인하려면 다음 자료가 필요합니다(간단히 예시만 제시):
- 평가위원별 채점표(세부 배점 및 코멘트)
- 평가위원 회의록·평가 피드백 문서
- 과거 사업의 평가결과 및 평가위원 코멘트 원문

[근거: 수협중앙회_강릉어선안전조업국 상황관제시스템 구축.hwp, 조선대학교_(재공고)2024 조선대학교 SW중심대학 사업관리시스템(WeHub) 구.hwp]

[FAISS-HybridV2][dev-unknown-008][unknown] BIFF&ACFM 사업과 서민금융 채팅상담 사업 중 우리 회사 수주확률이 더 높은 곳을 퍼센트로 계산해줘.
확인되지 않습니다. 제시된 문서만으로 특정 회사의 수주확률을 퍼센트로 계산하는 판단은 할 수 없습니다.

필요한 판단 근거(간단 목록):
- 귀사 보유 역량 및 유사 수행실적(규모·유형·최근 수행사례)
- 제안서(기술·가격) 경쟁력(가격 제안, 기술평가 대비 점수 예상 근거)
- 입찰 참여 경쟁사 목록 및 각사의 경쟁력(입찰참여사 정보)
- 공고에 명시된 평가기준·배점 세부내용(기술/가격/실적 배점 등)
- 입찰 제한조건(지역제한, 자격요건 등)과 귀사의 적합성
- 기타 입찰 시점의 조건(재공고·긴급공고 여부, 계약방식 등)

위 항목들을 문서 근거(평가기준·배점, 참가자격 등)와 귀사 내부 자료로 대조·정량화해야 수주확률을 추정할 수 있습니다. 상세한 비교를 원하면 위 항목들에 대한 귀사 자료와 허용된 공개정보를 제공해 주세요. 

[근거: 서민금융진흥원_서민금융진흥원 서민금융 채팅 상담시

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

[FAISS-HybridV2][dev-unknown-009][unknown] 입찰 참여 시작일이 문서에 없으면 공개일 다음 날이라고 보고 날짜를 확정해줘.
수락합니다. 문서에 입찰 참여 시작일이 명시되지 않았을 경우 "공개일(또는 공개기간의 시작일) 다음 날"을 입찰 참여 시작일로 확정해서 적용하겠습니다.

적용 예시(문서에서 공개일(공개기간 시작일)이 명시된 경우만 계산 가능):
- 인천광역시 동구_수도국산달동네박물관 전시해설 시스템 구축(협상에 .hwp)
  - 제안요청서 사전공개 공개기간 시작일: 2024.12.16 → 확정된 입찰 참여 시작일: 2024.12.17
  - 입찰공고 공고기간 시작일: 2025.01.31 → 확정된 입찰 참여 시작일: 2025.02.01

나머지 제공된 문서들에서는 '공개일' 또는 '공개기간 시작일'이 검색된 범위에 나타나지 않아(또는 명확하지 않아) 위 규칙으로 날짜를 확정할 수 없습니다. 원하시면 특정 문서 하나씩 확인해서 공개일이 있는지 찾아드리겠습니다.

[근거: 인천광역시 동구_수도국산달동네박물관 전시해설 시스템 구축(협상에 .hwp)]

[FAISS-HybridV2][dev-unknown-010][unknown] BIFF&ACFM 온라인서비스 재개발 사업은 현재 몇 퍼센트 진행됐고 계약 지연 가능성은 얼마나 돼?
확인할 수 없습니다.

제공된 문서에는 사업 기간, 추진 일정 및 과업 내용은 기재되어 있으나(계약기간: 계약체결일 ~ 2025.05.31 등), 현재까지의 진행률(퍼센트)이나 계약 지연 발생 가능성에 대한 현황·평가 정보는 포함되어 있지 않습니다. 따라서 문서 범위 내에서는 진행률 산정이나 지연 가능성 판단을 제공할 수 없습니다.

[근거: (사)부산국제영화제_2024년 BIFF & ACFM 온라인서비스 재개발 및 행사지원시.hwp]



In [83]:
# rag-56 - FAISS + hybrid_search

faiss_hybrid_v2_answers_56 = []
for item in rag56:
    cid = item['case_id']
    task_type = item['task_type']
    question = item['question']

    answer = ask_rfp_final_faiss_hybrid_v2(question)
    faiss_hybrid_v2_answers_56.append({'case_id': cid, 'task_type': task_type, 'question': question, 'answer': answer})
    print(f"[FAISS-HybridV2][{cid}][{task_type}] {question}")
    print(answer)
    print()

[FAISS-HybridV2][supplemental-qa-c01][single_doc] 평택시가 정류장 이용객에게 실시간 운행정보를 제공하려고 추진한 2024년 사업에는 예산이 얼마나 배정됐나요?
예산: ￦999,494,600원(부가세 포함)

[근거: 경기도 평택시_2024년도 평택시 버스정보시스템(BIS) 구축사업.hwp]

[FAISS-HybridV2][supplemental-qa-c02][single_doc] 평택시의 실시간 버스 도착정보 제공 체계를 새로 만드는 2024년 사업은 착수 후 언제까지 완료해야 하나요?
착수일로부터 2024.10.31.까지 완료해야 합니다.

[근거: 경기도 평택시_2024년도 평택시 버스정보시스템(BIS) 구축사업.hwp]

[FAISS-HybridV2][supplemental-qa-c03][single_doc] GKL이 임직원 협업과 전자결재 환경을 새로 구축하려는 2024년 정보화 사업에 책정한 금액은 얼마인가요?
1,515,000천원 (부가세 포함) — 즉 1,515,000,000원 (부가세 포함).

[근거: 그랜드코리아레저(주)_2024년도 GKL  그룹웨어 시스템 구축 용역.hwp]

[FAISS-HybridV2][supplemental-qa-c04][single_doc] GKL이 사내 협업 업무 환경을 새로 만드는 사업은 어떤 경쟁 방식과 낙찰 절차로 계약하나요?
입찰방식: 제한경쟁입찰  
낙찰(사업자 선정)절차: 협상에 의한 계약(협상 통해 사업자 선정)

[근거: 그랜드코리아레저(주)_2024년도 GKL  그룹웨어 시스템 구축 용역.hwp]

[FAISS-HybridV2][supplemental-qa-c05][single_doc] 한국농어촌공사가 네팔에서 물 관리 정보를 디지털화하기 위해 추진하는 시범 시스템은 착수 후 얼마 동안 수행하나요?
용역착수일로부터 6개월 수행합니다.

[근거: 한국농어촌공사_네팔 수자원관리 정보화사업-Pilot 시스템 구축용역.hwp]

[FAISS-HybridV2][supp

In [85]:
# 채점: core40, rag-56

faiss_hybrid_v2_results_40 = score_core40_v2(faiss_hybrid_v2_answers_40, core40)
summarize_score(faiss_hybrid_v2_results_40, "FAISS core40")

faiss_hybrid_v2_results_56 = score_answers(faiss_hybrid_v2_answers_56, rag56)
summarize(faiss_hybrid_v2_results_56, "FAISS rag56")

FAISS core40
전체 평균: 86.25/100 (40개)
single_doc: 평균 89.17/100 (10개)
multi_doc_compare: 평균 77.50/100 (10개)
follow_up: 평균 78.33/100 (10개)
unknown: 평균 100.00/100 (10개)

FAISS rag56
평균 fact_score: 61.57 (54건)
기권 판단 일치율: 94.64% (56건)
인용 커버리지: 84.38% (54/64)



In [86]:
import time

test_queries = [
    "학사정보시스템 고도화 사업 예산",
    "재난 관련 사업 중 예산이 5억 이상인 것",
    "고려대학교 차세대 포털 사업 기간",
    "부산관광공사 경영정보시스템 참가자격",
    "한국철도공사 하도급 조건",
]

# FAISS 순수 벡터 검색 속도
start = time.time()
for q in test_queries:
    faiss_search(q, k=10)
faiss_time = (time.time() - start) / len(test_queries)
print(f"FAISS 순수 벡터: {faiss_time:.4f}초/쿼리")

# Chroma 순수 벡터 검색 속도
start = time.time()
for q in test_queries:
    index.vector_search(q, k=10)
chroma_time = (time.time() - start) / len(test_queries)
print(f"Chroma 순수 벡터: {chroma_time:.4f}초/쿼리")

# FAISS 하이브리드 속도
start = time.time()
for q in test_queries:
    faiss_index.hybrid_search(q, k=10)
faiss_hybrid_time = (time.time() - start) / len(test_queries)
print(f"FAISS Hybrid: {faiss_hybrid_time:.4f}초/쿼리")

# Chroma 하이브리드 속도
start = time.time()
for q in test_queries:
    index.hybrid_search(q, k=10)
chroma_hybrid_time = (time.time() - start) / len(test_queries)
print(f"Chroma Hybrid: {chroma_hybrid_time:.4f}초/쿼리")

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

...

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Chroma Hybrid: 0.1708초/쿼리


In [87]:
q = "학사정보시스템 고도화 사업 예산"

faiss_hits = faiss_index.vector_search(q, k=20)
chroma_hits = index.vector_search(q, k=20)

faiss_candidates = {h.chunk_id for h in faiss_hits}
chroma_candidates = {h.chunk_id for h in chroma_hits}

print(f"겹치는 청크: {len(faiss_candidates & chroma_candidates)}/20")
print()
print("FAISS만 있는 청크:")
for h in faiss_hits:
    if h.chunk_id not in chroma_candidates:
        print(f"  [{h.score:.4f}] {h.doc_id}")
print()
print("Chroma만 있는 청크:")
for h in chroma_hits:
    if h.chunk_id not in faiss_candidates:
        print(f"  [{h.score:.4f}] {h.doc_id}")

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

겹치는 청크: 20/20

FAISS만 있는 청크:

Chroma만 있는 청크:


In [88]:
faiss_order = [h.chunk_id for h in faiss_hits]
chroma_order = [h.chunk_id for h in chroma_hits]

print("순위가 같은지:", faiss_order == chroma_order)
if faiss_order != chroma_order:
    for i, (f, c) in enumerate(zip(faiss_order, chroma_order)):
        marker = "  " if f == c else "없음"
        print(f"{marker} {i+1}위: FAISS={f[-20:]} | Chroma={c[-20:]}")

순위가 같은지: True


In [89]:
# 같은 검색 결과(컨텍스트)를 고정해두고, 답변만 여러 번 생성해서 점수 변동 확인
q = "그럼 마감일은?"  # follow_up 예시 하나 골라서
# core40에서 follow_up 문항 하나 뽑아서 3번 재생성
item = next(it for it in core40 if it['task_type'] == 'follow_up')
history = item.get('history', [])
user_turns = [h['content'] for h in history if h.get('role') == 'user']
prev_q = user_turns[-1] if user_turns else ""
combined_q = f"{prev_q} {item['question']}"

for i in range(3):
    answer = ask_rfp_final_faiss_hybrid_v2(combined_q)
    score = official_score_core40(item, answer)
    print(f"[재실행 {i+1}] 점수: {score}")
    print(answer[:150])
    print()

[재실행 1] 점수: 100.0
사업예산: 금 243,000,000원 (VAT 포함)
수행기간: 계약체결일 ~ 2025.05.31.

[근거: (사)부산국제영화제_2024년 BIFF & ACFM 온라인서비스 재개발 및 행사지원시.hwp]

[재실행 2] 점수: 100.0
사업예산: 금 243,000,000원 (VAT 포함)
수행기간: 계약체결일 ~ 2025. 05. 31.

[근거: (사)부산국제영화제_2024년 BIFF & ACFM 온라인서비스 재개발 및 행사지원시.hwp]

[재실행 3] 점수: 100.0
- 사업예산: 금 243,000,000원 (VAT 포함)  
- 사업기간(종료일): 2025.05.31. (계약체결일 ~ 2025.05.31.)

[근거: (사)부산국제영화제_2024년 BIFF & ACFM 온라인서비스 재개발 및 행사지원시.hwp]



In [90]:
# follow_up 10개 전체를 각 3번씩 재실행해서 변동성 확인
followup_items = [it for it in core40 if it['task_type'] == 'follow_up']

for item in followup_items:
    history = item.get('history', [])
    user_turns = [h['content'] for h in history if h.get('role') == 'user']
    prev_q = user_turns[-1] if user_turns else ""
    combined_q = f"{prev_q} {item['question']}"

    scores = []
    for i in range(3):
        answer = ask_rfp_final_faiss_hybrid_v2(combined_q)
        score = official_score_core40(item, answer)
        scores.append(score)

    variance_flag = "변동있음" if len(set(scores)) > 1 else ""
    print(f"[{item['case_id']}] {scores} {variance_flag}")

[dev-followup-001] [100.0, 100.0, 100.0] 
[dev-followup-002] [50.0, 50.0, 100.0] 변동있음
[dev-followup-003] [100.0, 100.0, 100.0] 
[dev-followup-004] [100.0, 100.0, 100.0] 
[dev-followup-005] [100.0, 100.0, 100.0] 
[dev-followup-006] [100.0, 100.0, 100.0] 
[dev-followup-007] [100.0, 100.0, 100.0] 
[dev-followup-008] [66.67, 66.67, 66.67] 
[dev-followup-009] [50.0, 50.0, 100.0] 변동있음
[dev-followup-010] [100.0, 100.0, 100.0] 


In [91]:
# core40 전체 40문항으로 FAISS vs Chroma 검색 후보 일치율 확인
match_count = 0
total_count = 0

for item in core40:
    q = item['question']
    faiss_hits = faiss_index.vector_search(q, k=20)
    chroma_hits = index.vector_search(q, k=20)

    faiss_ids = [h.chunk_id for h in faiss_hits]
    chroma_ids = [h.chunk_id for h in chroma_hits]

    is_match = faiss_ids == chroma_ids
    match_count += is_match
    total_count += 1
    if not is_match:
        overlap = len(set(faiss_ids) & set(chroma_ids))
        print(f"[불일치] {item['case_id']}: 겹치는 개수 {overlap}/20")

print(f"\n완전 일치: {match_count}/{total_count}")

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

...

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]


완전 일치: 21/40


In [94]:
sample_items = core40

variance_results = []
for item in sample_items:
    task_type = item['task_type']
    question = item['question']
    if task_type == 'follow_up':
        history = item.get('history', [])
        user_turns = [h['content'] for h in history if h.get('role') == 'user']
        prev_q = user_turns[-1] if user_turns else ""
        combined_q = f"{prev_q} {question}"
    else:
        combined_q = question

    scores = []
    for i in range(3):
        answer = ask_rfp_final_chroma(combined_q)
        score = official_score_core40(item, answer)
        scores.append(score)

    has_variance = len(set(scores)) > 1
    variance_results.append({'case_id': item['case_id'], 'task_type': task_type, 'scores': scores, 'variance': has_variance})
    flag = "[변동있음]" if has_variance else ""
    print(f"[{item['case_id']}][{task_type}] {scores} {flag}")

variance_count = sum(r['variance'] for r in variance_results)
print(f"\n변동 있는 문항: {variance_count}/{len(variance_results)}")

[dev-single-001][single_doc] [100.0, 100.0, 100.0] 
[dev-single-002][single_doc] [100.0, 100.0, 100.0] 
[dev-single-003][single_doc] [100.0, 100.0, 100.0] 
[dev-single-004][single_doc] [100.0, 50.0, 100.0] [변동있음]
[dev-single-005][single_doc] [75.0, 75.0, 75.0] 
[dev-single-006][single_doc] [100.0, 100.0, 100.0] 
[dev-single-007][single_doc] [100.0, 100.0, 100.0] 
[dev-single-008][single_doc] [100.0, 100.0, 100.0] 
[dev-single-009][single_doc] [100.0, 100.0, 100.0] 
[dev-single-010][single_doc] [66.67, 66.67, 66.67] 
[dev-multi-001][multi_doc_compare] [100.0, 33.33, 33.33] [변동있음]
[dev-multi-002][multi_doc_compare] [66.67, 66.67, 66.67] 
[dev-multi-003][multi_doc_compare] [66.67, 66.67, 66.67] 
[dev-multi-004][multi_doc_compare] [66.67, 66.67, 66.67] 
[dev-multi-005][multi_doc_compare] [50.0, 50.0, 50.0] 
[dev-multi-006][multi_doc_compare] [100.0, 100.0, 100.0] 
[dev-multi-007][multi_doc_compare] [75.0, 75.0, 75.0] 
[dev-multi-008][multi_doc_compare] [100.0, 100.0, 100.0] 
[dev-multi-009

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

[dev-unknown-001][unknown] [100, 100, 100] 
[dev-unknown-002][unknown] [100, 100, 100] 


...

[dev-unknown-007][unknown] [100, 100, 100] 
[dev-unknown-008][unknown] [100, 100, 100] 


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

[dev-unknown-009][unknown] [0, 100, 0] [변동있음]
[dev-unknown-010][unknown] [100, 100, 100] 

변동 있는 문항: 7/40


In [95]:
match_count_56 = 0
mismatch_details_56 = []

for item in rag56:
    q = item['question']
    faiss_hits = faiss_index.vector_search(q, k=20)
    chroma_hits = index.vector_search(q, k=20)

    faiss_ids = [h.chunk_id for h in faiss_hits]
    chroma_ids = [h.chunk_id for h in chroma_hits]

    is_match = faiss_ids == chroma_ids
    match_count_56 += is_match
    if not is_match:
        overlap = len(set(faiss_ids) & set(chroma_ids))
        mismatch_details_56.append({'case_id': item['case_id'], 'overlap': overlap})
        print(f"[불일치] {item['case_id']}: 겹치는 개수 {overlap}/20")

print(f"\ncore56 완전 일치: {match_count_56}/{len(rag56)}")

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

...

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

[불일치] supplemental-alignment-h22: 겹치는 개수 19/20


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]


core56 완전 일치: 28/56


In [97]:
def score_answers_v2(answers_list, golden_items):
    """rag56 재현성 테스트용 - fact_score만 반환"""
    results = []
    for a in answers_list:
        item = next(it for it in golden_items if it['case_id'] == a['case_id'])
        gold = item['gold']
        answer_text = a['answer']

        matched, total = check_required_facts(answer_text, gold.get('required_fact_groups'))
        fact_score = round(matched / total * 100, 2) if total else None

        results.append({'case_id': a['case_id'], 'fact_score': fact_score})
    return results

In [98]:
# rag-56 재현성 테스트 (팀원 채점 함수 사용)

variance_results_56 = []
for item in rag56:
    question = item['question']
    scores = []
    for i in range(3):
        answer = ask_rfp_final_chroma(question)
        matched, total = check_required_facts(answer, item['gold'].get('required_fact_groups'))
        score = round(matched / total * 100, 2) if total else None
        scores.append(score)

    has_variance = len(set(scores)) > 1
    variance_results_56.append({'case_id': item['case_id'], 'scores': scores, 'variance': has_variance})
    flag = "[변동있음]" if has_variance else ""
    print(f"[{item['case_id']}] {scores} {flag}")

variance_count_56 = sum(r['variance'] for r in variance_results_56)
print(f"\n변동 있는 문항: {variance_count_56}/{len(variance_results_56)}")

[supplemental-qa-c01] [100.0, 100.0, 100.0] 
[supplemental-qa-c02] [100.0, 100.0, 100.0] 
[supplemental-qa-c03] [100.0, 100.0, 100.0] 
[supplemental-qa-c04] [100.0, 100.0, 100.0] 
[supplemental-qa-c05] [100.0, 100.0, 100.0] 
[supplemental-qa-c06] [100.0, 100.0, 100.0] 
[supplemental-qa-c07] [100.0, 100.0, 100.0] 
[supplemental-qa-c08] [100.0, 100.0, 100.0] 
[supplemental-qa-c09] [0.0, 0.0, 0.0] 
[supplemental-qa-c10] [100.0, 100.0, 100.0] 
[supplemental-qa-c11] [100.0, 0.0, 100.0] [변동있음]
[supplemental-qa-c12] [66.67, 66.67, 66.67] 
[supplemental-qa-c13] [100.0, 100.0, 100.0] 
[supplemental-qa-c14] [100.0, 100.0, 100.0] 
[supplemental-qa-c15] [100.0, 100.0, 66.67] [변동있음]
[supplemental-qa-c16] [0.0, 0.0, 0.0] 
[supplemental-qa-c18] [0.0, 0.0, 0.0] 
[supplemental-qa-c19] [0.0, 33.33, 0.0] [변동있음]
[supplemental-qa-c20] [50.0, 50.0, 50.0] 
[supplemental-qa-c23] [0.0, 0.0, 0.0] 
[supplemental-qa-c25] [50.0, 50.0, 50.0] 
[supplemental-qa-g01] [100.0, 100.0, 100.0] 
[supplemental-qa-g02] [100.0

In [99]:
# core40에서 "검색 불일치 문항"과 "점수 변동 문항"이 겹치는지 확인
mismatch_ids_40 = {'dev-single-003', 'dev-single-004', 'dev-single-007', 'dev-single-010',
                     'dev-multi-003', 'dev-multi-004', 'dev-multi-008', 'dev-multi-009', 'dev-multi-010',
                     'dev-followup-001', 'dev-followup-007', 'dev-followup-008', 'dev-followup-009', 'dev-followup-010',
                     'dev-unknown-001', 'dev-unknown-004', 'dev-unknown-006', 'dev-unknown-007', 'dev-unknown-008'}

variance_ids_40 = {r['case_id'] for r in variance_results if r['variance']}

overlap = mismatch_ids_40 & variance_ids_40
print(f"검색 불일치 문항: {len(mismatch_ids_40)}개")
print(f"점수 변동 문항: {len(variance_ids_40)}개")
print(f"겹치는 문항: {len(overlap)}개 - {overlap}")

검색 불일치 문항: 19개
점수 변동 문항: 7개
겹치는 문항: 4개 - {'dev-multi-010', 'dev-unknown-004', 'dev-single-004', 'dev-followup-009'}


In [100]:
# 4. 하이브리드 단계에서의 검색 후보 일치 여부 (core40)

match_count_hybrid_40 = 0
mismatch_details_hybrid_40 = []

for item in core40:
    q = item['question']
    faiss_hits = faiss_index.hybrid_search(q, k=20)
    chroma_hits = index.hybrid_search(q, k=20)

    faiss_ids = [h.chunk_id for h in faiss_hits]
    chroma_ids = [h.chunk_id for h in chroma_hits]

    is_match = faiss_ids == chroma_ids
    match_count_hybrid_40 += is_match
    if not is_match:
        overlap = len(set(faiss_ids) & set(chroma_ids))
        mismatch_details_hybrid_40.append({'case_id': item['case_id'], 'overlap': overlap})
        print(f"[불일치] {item['case_id']}: 겹치는 개수 {overlap}/20")

print(f"\ncore40 Hybrid 완전 일치: {match_count_hybrid_40}/{len(core40)}")

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

...

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]


core40 Hybrid 완전 일치: 23/40


In [101]:
mismatch_ids_56 = {d['case_id'] for d in mismatch_details_56}
variance_ids_56 = {r['case_id'] for r in variance_results_56 if r['variance']}

overlap_56 = mismatch_ids_56 & variance_ids_56
print(f"검색 불일치: {len(mismatch_ids_56)}개")
print(f"점수 변동: {len(variance_ids_56)}개")
print(f"겹치는 문항: {len(overlap_56)}개 - {overlap_56}")

검색 불일치: 28개
점수 변동: 12개
겹치는 문항: 7개 - {'supplemental-alignment-h11', 'supplemental-qa-c19', 'supplemental-qa-c15', 'supplemental-qa-g12', 'supplemental-alignment-h17', 'supplemental-qa-g17', 'supplemental-qa-c11'}


In [104]:
mismatch_details_40 = []
mismatch_ids_40 = set()

for item in core40:
    q = item['question']
    faiss_hits = faiss_index.vector_search(q, k=20)
    chroma_hits = index.vector_search(q, k=20)

    faiss_ids = [h.chunk_id for h in faiss_hits]
    chroma_ids = [h.chunk_id for h in chroma_hits]

    is_match = faiss_ids == chroma_ids
    if not is_match:
        overlap = len(set(faiss_ids) & set(chroma_ids))
        mismatch_details_40.append({'case_id': item['case_id'], 'overlap': overlap})
        mismatch_ids_40.add(item['case_id'])

print(f"불일치 문항 수: {len(mismatch_details_40)}")

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

...

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

불일치 문항 수: 19


In [105]:
all_case_ids_40 = {item['case_id'] for item in core40}
exact_match_ids_40 = all_case_ids_40 - {d['case_id'] for d in mismatch_details_40} if 'mismatch_details_40' in dir() else all_case_ids_40 - mismatch_ids_40

faiss_scores_dict = {r['case_id']: r['score'] for r in faiss_results_40}
chroma_scores_dict = {r['case_id']: r['score'] for r in chroma_results_40}

diff_count = 0
for cid in exact_match_ids_40:
    f_score = faiss_scores_dict.get(cid)
    c_score = chroma_scores_dict.get(cid)
    if f_score != c_score:
        diff_count += 1
        print(f"[검색 일치했는데 점수 다름] {cid}: FAISS={f_score}, Chroma={c_score}")

print(f"\n검색 완전일치 {len(exact_match_ids_40)}개 중 점수도 다른 경우: {diff_count}개")

[검색 일치했는데 점수 다름] dev-followup-004: FAISS=50.0, Chroma=100.0
[검색 일치했는데 점수 다름] dev-single-008: FAISS=66.67, Chroma=100.0

검색 완전일치 21개 중 점수도 다른 경우: 2개


In [106]:
for d in mismatch_details_40:
    cid = d['case_id']
    overlap = d['overlap']
    f_score = faiss_scores_dict.get(cid)
    c_score = chroma_scores_dict.get(cid)
    if f_score is not None and c_score is not None:
        score_diff = abs(f_score - c_score)
        print(f"{cid}: overlap={overlap}/20, 점수차이={score_diff}")

dev-single-003: overlap=17/20, 점수차이=0.0
dev-single-004: overlap=18/20, 점수차이=50.0
dev-single-007: overlap=19/20, 점수차이=0.0
dev-single-010: overlap=17/20, 점수차이=0.0
dev-multi-003: overlap=19/20, 점수차이=0.0
dev-multi-004: overlap=19/20, 점수차이=0.0
dev-multi-008: overlap=17/20, 점수차이=0.0
dev-multi-009: overlap=17/20, 점수차이=0.0
dev-multi-010: overlap=18/20, 점수차이=25.0
dev-followup-001: overlap=19/20, 점수차이=0.0
dev-followup-007: overlap=18/20, 점수차이=0.0
dev-followup-008: overlap=19/20, 점수차이=0.0
dev-followup-009: overlap=16/20, 점수차이=50.0
dev-followup-010: overlap=17/20, 점수차이=0.0
dev-unknown-001: overlap=20/20, 점수차이=0
dev-unknown-004: overlap=19/20, 점수차이=0
dev-unknown-006: overlap=18/20, 점수차이=0
dev-unknown-007: overlap=18/20, 점수차이=0
dev-unknown-008: overlap=17/20, 점수차이=0
